In [1]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, RobustScaler

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 2 — LOAD TRANSACTION DATA
# ============================================================

tx_features = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\txs_features.csv")
tx_classes = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\txs_classes.csv")

print("Transaction features shape :", tx_features.shape)
print("Transaction classes shape :", tx_classes.shape)

Transaction features shape : (203769, 184)
Transaction classes shape : (203769, 2)


In [3]:
# ============================================================
# CELL 3 — MERGE FEATURES AND LABELS
# ============================================================

tx_data = tx_features.merge(
    tx_classes,
    on="txId",
    how="inner"
)

print("Merged dataset shape:", tx_data.shape)

display(tx_data.head())

Merged dataset shape: (203769, 185)


,txId,Time step,Local_feature_1,Local_feature_2,Local_feature_3,Local_feature_4,Local_feature_5,Local_feature_6,Local_feature_7,Local_feature_8,Local_feature_9,Local_feature_10,Local_feature_11,Local_feature_12,Local_feature_13,Local_feature_14,Local_feature_15,Local_feature_16,Local_feature_17,Local_feature_18,Local_feature_19,Local_feature_20,Local_feature_21,Local_feature_22,Local_feature_23,Local_feature_24,Local_feature_25,Local_feature_26,Local_feature_27,Local_feature_28,Local_feature_29,Local_feature_30,Local_feature_31,Local_feature_32,Local_feature_33,Local_feature_34,Local_feature_35,Local_feature_36,Local_feature_37,Local_feature_38,Local_feature_39,Local_feature_40,Local_feature_41,Local_feature_42,Local_feature_43,Local_feature_44,Local_feature_45,Local_feature_46,Local_feature_47,Local_feature_48,Local_feature_49,Local_feature_50,Local_feature_51,Local_feature_52,Local_feature_53,Local_feature_54,Local_feature_55,Local_feature_56,Local_feature_57,Local_feature_58,Local_feature_59,Local_feature_60,Local_feature_61,Local_feature_62,Local_feature_63,Local_feature_64,Local_feature_65,Local_feature_66,Local_feature_67,Local_feature_68,Local_feature_69,Local_feature_70,Local_feature_71,Local_feature_72,Local_feature_73,Local_feature_74,Local_feature_75,Local_feature_76,Local_feature_77,Local_feature_78,Local_feature_79,Local_feature_80,Local_feature_81,Local_feature_82,Local_feature_83,Local_feature_84,Local_feature_85,Local_feature_86,Local_feature_87,Local_feature_88,Local_feature_89,Local_feature_90,Local_feature_91,Local_feature_92,Local_feature_93,Aggregate_feature_1,Aggregate_feature_2,Aggregate_feature_3,Aggregate_feature_4,Aggregate_feature_5,Aggregate_feature_6,Aggregate_feature_7,Aggregate_feature_8,Aggregate_feature_9,Aggregate_feature_10,Aggregate_feature_11,Aggregate_feature_12,Aggregate_feature_13,Aggregate_feature_14,Aggregate_feature_15,Aggregate_feature_16,Aggregate_feature_17,Aggregate_feature_18,Aggregate_feature_19,Aggregate_feature_20,Aggregate_feature_21,Aggregate_feature_22,Aggregate_feature_23,Aggregate_feature_24,Aggregate_feature_25,Aggregate_feature_26,Aggregate_feature_27,Aggregate_feature_28,Aggregate_feature_29,Aggregate_feature_30,Aggregate_feature_31,Aggregate_feature_32,Aggregate_feature_33,Aggregate_feature_34,Aggregate_feature_35,Aggregate_feature_36,Aggregate_feature_37,Aggregate_feature_38,Aggregate_feature_39,Aggregate_feature_40,Aggregate_feature_41,Aggregate_feature_42,Aggregate_feature_43,Aggregate_feature_44,Aggregate_feature_45,Aggregate_feature_46,Aggregate_feature_47,Aggregate_feature_48,Aggregate_feature_49,Aggregate_feature_50,Aggregate_feature_51,Aggregate_feature_52,Aggregate_feature_53,Aggregate_feature_54,Aggregate_feature_55,Aggregate_feature_56,Aggregate_feature_57,Aggregate_feature_58,Aggregate_feature_59,Aggregate_feature_60,Aggregate_feature_61,Aggregate_feature_62,Aggregate_feature_63,Aggregate_feature_64,Aggregate_feature_65,Aggregate_feature_66,Aggregate_feature_67,Aggregate_feature_68,Aggregate_feature_69,Aggregate_feature_70,Aggregate_feature_71,Aggregate_feature_72,in_txs_degree,out_txs_degree,total_BTC,fees,size,num_input_addresses,num_output_addresses,in_BTC_min,in_BTC_max,in_BTC_mean,in_BTC_median,in_BTC_total,out_BTC_min,out_BTC_max,out_BTC_mean,out_BTC_median,out_BTC_total,class
0,3321,1,-0.169615,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.160199,-0.166062,-0.049707,-0.162507,-0.028741,-0.035391,-0.042955,-0.013282,-0.042183,-0.168770,-0.171416,-0.172277,-1.373657,-1.371460,-0.139663,-0.148869,-0.080147,-0.155604,-0.010763,-0.012107,-0.139665,-0.148864,-0.080147,-0.155604,-0.010669,-0.012005,-0.024668,-0.031272,-0.023045,-0.026214,0.001428,0.001483,-0.226298,-0.238504,-0.075256,-0.234053,0.037468,0.043444,-0.226681,-0.242726,-0.097895,-0.235378,0.036577,0.042345,1.830780,1.378639,-0.232553,1.686410,0.048767,0.052956,-0.038877,-0.172804,-0.163071,-0.160801,-1.316342,-1.315388,-0.038872,-0.172793,-0.163060,-0.160794,-1.316333,-1.31537

In [4]:
# ============================================================
# CELL 4 — CHECK DUPLICATE TRANSACTIONS
# ============================================================

duplicate_txids = tx_data["txId"].duplicated().sum()

print("Total transactions        :", len(tx_data))
print("Duplicate txId entries    :", duplicate_txids)
print("Unique txId values        :", tx_data["txId"].nunique())

if duplicate_txids == 0:
    print("\n✅ No duplicate transaction IDs found.")
else:
    print("\n⚠️ Duplicate transaction IDs found.")

Total transactions        : 203769
Duplicate txId entries    : 0
Unique txId values        : 203769

✅ No duplicate transaction IDs found.


In [5]:
# ============================================================
# CELL 5 — CHECK DUPLICATE ROWS
# ============================================================

duplicate_rows = tx_data.duplicated().sum()

print("Total rows              :", len(tx_data))
print("Completely duplicate rows:", duplicate_rows)

if duplicate_rows == 0:
    print("\n✅ No completely duplicated rows found.")
else:
    print("\n⚠️ Completely duplicated rows found.")

Total rows              : 203769
Completely duplicate rows: 0

✅ No completely duplicated rows found.


In [6]:
# ============================================================
# CELL 6 — SEPARATE ID, FEATURES AND TARGET
# ============================================================

# Transaction identifier
tx_ids = tx_data["txId"].copy()

# Target label
y = tx_data["class"].copy()

# All feature columns
feature_cols = [
    col for col in tx_data.columns
    if col not in ["txId", "class"]
]

X = tx_data[feature_cols].copy()

print("Transaction IDs :", len(tx_ids))
print("Feature columns  :", len(feature_cols))
print("Feature matrix   :", X.shape)
print("Target           :", y.shape)

Transaction IDs : 203769
Feature columns  : 183
Feature matrix   : (203769, 183)
Target           : (203769,)


In [7]:
# ============================================================
# CELL 7 — VERIFY FEATURE MATRIX
# ============================================================

print("First 10 features:")
print(feature_cols[:10])

print("\nLast 10 features:")
print(feature_cols[-10:])

print("\nMissing values in X:", X.isna().sum().sum())
print("Unique target classes:", sorted(y.unique()))

print("\nFeature matrix data types:")
print(X.dtypes.value_counts())

First 10 features:
['Time step', 'Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9']

Last 10 features:
['in_BTC_min', 'in_BTC_max', 'in_BTC_mean', 'in_BTC_median', 'in_BTC_total', 'out_BTC_min', 'out_BTC_max', 'out_BTC_mean', 'out_BTC_median', 'out_BTC_total']

Missing values in X: 16405
Unique target classes: [np.int64(1), np.int64(2), np.int64(3)]

Feature matrix data types:
float64    182
int64        1
Name: count, dtype: int64


In [8]:
# ============================================================
# CELL 8 — IDENTIFY FEATURES WITH MISSING VALUES
# ============================================================

missing_counts = X.isna().sum()

missing_features = missing_counts[missing_counts > 0].sort_values(
    ascending=False
)

print("Number of features with missing values:", len(missing_features))
print("\nMissing-value summary:")

display(
    missing_features.to_frame("missing_count")
)

Number of features with missing values: 17

Missing-value summary:


,missing_count
in_txs_degree,965
out_txs_degree,965
total_BTC,965
fees,965
size,965
num_input_addresses,965
num_output_addresses,965
in_BTC_min,965
in_BTC_max,965
in_BTC_mean,965


In [9]:
# ============================================================
# CELL 9 — MISSING VALUES BY FEATURE GROUP
# ============================================================

augmented_features = [
    "in_txs_degree",
    "out_txs_degree",
    "total_BTC",
    "fees",
    "size",
    "num_input_addresses",
    "num_output_addresses",
    "in_BTC_min",
    "in_BTC_max",
    "in_BTC_mean",
    "in_BTC_median",
    "in_BTC_total",
    "out_BTC_min",
    "out_BTC_max",
    "out_BTC_mean",
    "out_BTC_median",
    "out_BTC_total"
]

local_features = [
    f"Local_feature_{i}" for i in range(1, 94)
]

aggregate_features = [
    f"Aggregate_feature_{i}" for i in range(1, 73)
]

missing_summary = pd.DataFrame({
    "Feature": feature_cols,
    "Missing_Count": X.isna().sum().values
})

def get_feature_group(feature):
    if feature == "Time step":
        return "Time step"
    elif feature in local_features:
        return "Local"
    elif feature in aggregate_features:
        return "Aggregate"
    elif feature in augmented_features:
        return "Elliptic++ augmented"
    else:
        return "Other"

missing_summary["Group"] = missing_summary["Feature"].apply(
    get_feature_group
)

display(
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)

,Feature,Missing_Count,Group
166,in_txs_degree,965,Elliptic++ augmented
167,out_txs_degree,965,Elliptic++ augmented
168,total_BTC,965,Elliptic++ augmented
169,fees,965,Elliptic++ augmented
170,size,965,Elliptic++ augmented
171,num_input_addresses,965,Elliptic++ augmented
172,num_output_addresses,965,Elliptic++ augmented
173,in_BTC_min,965,Elliptic++ augmented
174,in_BTC_max,965,Elliptic++ augmented
175,in_BTC_mean,965,Elliptic++ augmented


In [10]:
# ============================================================
# CELL 10 — IDENTIFY TRANSACTIONS WITH AUGMENTED FEATURES MISSING
# ============================================================

# Check whether all 17 augmented features are missing
augmented_missing_mask = X[augmented_features].isna().all(axis=1)

affected_tx = tx_data.loc[
    augmented_missing_mask,
    ["txId", "Time step", "class"]
].copy()

print("Transactions with all augmented features missing:",
      len(affected_tx))

print("\nClass distribution:")
print(affected_tx["class"].value_counts().sort_index())

print("\nTime-step distribution:")
display(
    affected_tx["Time step"]
    .value_counts()
    .sort_index()
    .to_frame("transaction_count")
)

Transactions with all augmented features missing: 965

Class distribution:
class
2    519
3    446
Name: count, dtype: int64

Time-step distribution:


,transaction_count
Time step,
1,36
2,15
4,4
5,2
7,22
8,26
9,5
11,7
12,8


In [11]:
# ============================================================
# CELL 11 — COMPARE AFFECTED AND UNAFFECTED TRANSACTIONS
# ============================================================

comparison = pd.DataFrame({
    "Group": ["Augmented features available",
              "All augmented features missing"],
    "Transaction_Count": [
        (~augmented_missing_mask).sum(),
        augmented_missing_mask.sum()
    ],
    "Mean_Time_Step": [
        tx_data.loc[~augmented_missing_mask, "Time step"].mean(),
        tx_data.loc[augmented_missing_mask, "Time step"].mean()
    ],
    "Median_Time_Step": [
        tx_data.loc[~augmented_missing_mask, "Time step"].median(),
        tx_data.loc[augmented_missing_mask, "Time step"].median()
    ]
})

display(comparison)

print("\nClass distribution among affected transactions:")
display(
    pd.crosstab(
        affected_tx["class"],
        columns="count"
    )
)

,Group,Transaction_Count,Mean_Time_Step,Median_Time_Step
0,Augmented features available,202804,23.807213,23.0
1,All augmented features missing,965,31.566839,36.0



Class distribution among affected transactions:


col_0,count
class,
2,519
3,446


In [12]:
# ============================================================
# CELL 12 — CHECK TRANSACTION GRAPH COVERAGE
# ============================================================

tx_edges = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\txs_edgelist.csv")

affected_ids = set(affected_tx["txId"])

# Transactions appearing anywhere in the transaction graph
graph_tx_ids = set(tx_edges["txId1"]) | set(tx_edges["txId2"])

affected_in_graph = affected_tx["txId"].isin(graph_tx_ids)

print("Affected transactions                  :", len(affected_tx))
print("Affected transactions in tx graph     :", affected_in_graph.sum())
print("Affected transactions absent from graph:", (~affected_in_graph).sum())

Affected transactions                  : 965
Affected transactions in tx graph     : 965
Affected transactions absent from graph: 0


In [14]:
# ============================================================
# CELL 13 — CHECK ADDRESS GRAPH COVERAGE
# ============================================================

addr_tx = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\AddrTx_edgelist.csv")
tx_addr = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\TxAddr_edgelist.csv")

input_tx_ids = set(addr_tx["txId"])
output_tx_ids = set(tx_addr["txId"])

affected_with_input_addresses = affected_tx["txId"].isin(input_tx_ids)
affected_with_output_addresses = affected_tx["txId"].isin(output_tx_ids)

print("Affected transactions with input-address records :",
      affected_with_input_addresses.sum())

print("Affected transactions with output-address records:",
      affected_with_output_addresses.sum())

print("Affected transactions with no address records    :",
      (
          ~affected_with_input_addresses &
          ~affected_with_output_addresses
      ).sum()
      )

Affected transactions with input-address records : 0
Affected transactions with output-address records: 0
Affected transactions with no address records    : 965


In [15]:
# ============================================================
# CELL 10 — MISSINGNESS IN LABELLED DATA
# ============================================================

labelled_mask = tx_data["class"].isin([1, 2])

labelled_missing = (
    augmented_missing_mask & labelled_mask
)

print("Total labelled transactions:", labelled_mask.sum())
print("Labelled transactions with missing augmented features:",
      labelled_missing.sum())

print("\nClass distribution:")
print(
    tx_data.loc[labelled_missing, "class"]
    .value_counts()
    .sort_index()
)

Total labelled transactions: 46564
Labelled transactions with missing augmented features: 519

Class distribution:
class
2    519
Name: count, dtype: int64


In [16]:
# ============================================================
# CELL 11 — COMPARE LABELLED GROUPS
# ============================================================

labelled_available = labelled_mask & ~augmented_missing_mask
labelled_missing = labelled_mask & augmented_missing_mask

comparison = pd.DataFrame({
    "Group": [
        "Labelled + augmented features available",
        "Labelled + augmented features missing"
    ],
    "Count": [
        labelled_available.sum(),
        labelled_missing.sum()
    ],
    "Illicit": [
        (tx_data.loc[labelled_available, "class"] == 1).sum(),
        (tx_data.loc[labelled_missing, "class"] == 1).sum()
    ],
    "Licit": [
        (tx_data.loc[labelled_available, "class"] == 2).sum(),
        (tx_data.loc[labelled_missing, "class"] == 2).sum()
    ]
})

display(comparison)

,Group,Count,Illicit,Licit
0,Labelled + augmented features available,46045,4545,41500
1,Labelled + augmented features missing,519,0,519


In [17]:
# ============================================================
# CELL 12 — CHECK INHERITED FEATURES FOR LABELLED DATA
# ============================================================

inherited_features = (
    ["Time step"] +
    [f"Local_feature_{i}" for i in range(1, 94)] +
    [f"Aggregate_feature_{i}" for i in range(1, 73)]
)

labelled_data = tx_data[tx_data["class"].isin([1, 2])].copy()

inherited_missing = labelled_data[inherited_features].isna().sum()

print("Inherited features:", len(inherited_features))
print("Total missing values in inherited features:",
      inherited_missing.sum())

print("\nFeatures with missing values:")
display(
    inherited_missing[inherited_missing > 0]
    .to_frame("missing_count")
)

Inherited features: 166
Total missing values in inherited features: 0

Features with missing values:


,missing_count


In [18]:
# ============================================================
# CELL 13 — CREATE AUGMENTED FEATURE MISSINGNESS INDICATOR
# ============================================================

X_preprocessed = X.copy()

# 1 = augmented address-related information unavailable
# 0 = augmented information available
X_preprocessed["augmented_features_missing"] = (
    X_preprocessed[augmented_features]
    .isna()
    .all(axis=1)
    .astype(int)
)

print("Missingness indicator distribution:")
print(
    X_preprocessed["augmented_features_missing"]
    .value_counts()
)

Missingness indicator distribution:
augmented_features_missing
0    202804
1       965
Name: count, dtype: int64


In [19]:
# ============================================================
# CELL 14 — CREATE LABELLED MODELLING DATASET
# ============================================================

labelled_data = tx_data[
    tx_data["class"].isin([1, 2])
].copy()

# Binary target:
# 0 = Licit
# 1 = Illicit
labelled_data["target"] = (
    labelled_data["class"] == 1
).astype(int)

print("Labelled dataset shape:", labelled_data.shape)

print("\nTarget distribution:")
print(
    labelled_data["target"]
    .value_counts()
    .sort_index()
)

Labelled dataset shape: (46564, 186)

Target distribution:
target
0    42019
1     4545
Name: count, dtype: int64


In [20]:
# ============================================================
# CELL 15 — SEPARATE FEATURES, TARGET AND IDs
# ============================================================

model_feature_cols = [
    col for col in tx_features.columns
    if col != "txId"
]

X_labelled = labelled_data[model_feature_cols].copy()
y_labelled = labelled_data["target"].copy()
labelled_tx_ids = labelled_data["txId"].copy()

print("X shape       :", X_labelled.shape)
print("y shape       :", y_labelled.shape)
print("Feature count :", len(model_feature_cols))
print("Target values :", sorted(y_labelled.unique()))

X shape       : (46564, 183)
y shape       : (46564,)
Feature count : 183
Target values : [np.int64(0), np.int64(1)]


In [21]:
# ============================================================
# CELL 16 — SORT LABELLED DATA BY TIME STEP
# ============================================================

labelled_data = labelled_data.sort_values(
    by="Time step"
).reset_index(drop=True)

print("Labelled data sorted by Time step.")

print("\nTime-step range:")
print("Minimum:", labelled_data["Time step"].min())
print("Maximum:", labelled_data["Time step"].max())

display(
    labelled_data[["txId", "Time step", "class", "target"]].head()
)

Labelled data sorted by Time step.

Time-step range:
Minimum: 1
Maximum: 49


,txId,Time step,class,target
0,3403739,1,2,0
1,3403752,1,2,0
2,3403763,1,2,0
3,3404363,1,2,0
4,3405005,1,2,0


In [22]:
# ============================================================
# CELL 17 — TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

unique_time_steps = sorted(
    labelled_data["Time step"].unique()
)

n_steps = len(unique_time_steps)

train_end = int(n_steps * 0.70)
val_end = int(n_steps * 0.85)

train_steps = unique_time_steps[:train_end]
val_steps = unique_time_steps[train_end:val_end]
test_steps = unique_time_steps[val_end:]

train_data = labelled_data[
    labelled_data["Time step"].isin(train_steps)
].copy()

val_data = labelled_data[
    labelled_data["Time step"].isin(val_steps)
].copy()

test_data = labelled_data[
    labelled_data["Time step"].isin(test_steps)
].copy()

print("Train time steps:", train_steps[0], "to", train_steps[-1])
print("Validation time steps:", val_steps[0], "to", val_steps[-1])
print("Test time steps:", test_steps[0], "to", test_steps[-1])

print("\nDataset sizes:")
print("Train      :", train_data.shape)
print("Validation :", val_data.shape)
print("Test       :", test_data.shape)

Train time steps: 1 to 34
Validation time steps: 35 to 41
Test time steps: 42 to 49

Dataset sizes:
Train      : (29894, 186)
Validation : (7829, 186)
Test       : (8841, 186)


In [23]:
# ============================================================
# CELL 18 — CREATE TRAIN / VALIDATION / TEST MATRICES
# ============================================================

X_train = train_data[model_feature_cols].copy()
y_train = train_data["target"].copy()

X_val = val_data[model_feature_cols].copy()
y_val = val_data["target"].copy()

X_test = test_data[model_feature_cols].copy()
y_test = test_data["target"].copy()

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

X_train: (29894, 183)
X_val  : (7829, 183)
X_test : (8841, 183)

y_train: (29894,)
y_val  : (7829,)
y_test : (8841,)


In [24]:
# ============================================================
# CELL 19 — CLASS DISTRIBUTION AFTER TEMPORAL SPLIT
# ============================================================

split_distribution = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Validation": y_val.value_counts(),
    "Test": y_test.value_counts()
}).fillna(0).astype(int)

split_percentage = (
    split_distribution
    .div(split_distribution.sum(axis=0), axis=1)
    .mul(100)
    .round(2)
)

print("Class counts:")
display(split_distribution)

print("\nClass percentages:")
display(split_percentage)

Class counts:


,Train,Validation,Test
target,,,
0,26432,7154,8433
1,3462,675,408



Class percentages:


,Train,Validation,Test
target,,,
0,88.42,91.38,95.39
1,11.58,8.62,4.61


In [25]:
# ============================================================
# CELL 20 — CHECK MISSING VALUES IN EACH SPLIT
# ============================================================

print("Missing values in training set:",
      X_train.isna().sum().sum())

print("Missing values in validation set:",
      X_val.isna().sum().sum())

print("Missing values in test set:",
      X_test.isna().sum().sum())

print("\nMissing features in training set:")
display(
    X_train.isna().sum()
    .loc[lambda x: x > 0]
    .to_frame("missing_count")
)

Missing values in training set: 3315
Missing values in validation set: 3774
Missing values in test set: 1734

Missing features in training set:


,missing_count
in_txs_degree,195
out_txs_degree,195
total_BTC,195
fees,195
size,195
num_input_addresses,195
num_output_addresses,195
in_BTC_min,195
in_BTC_max,195
in_BTC_mean,195


In [26]:
# ============================================================
# CELL 21 — MISSING AUGMENTED FEATURES BY SPLIT
# ============================================================

for name, data in [
    ("Train", X_train),
    ("Validation", X_val),
    ("Test", X_test)
]:
    
    missing_rows = data[augmented_features].isna().all(axis=1).sum()
    
    print(f"{name}:")
    print(f"  Rows with all 17 augmented features missing: {missing_rows}")

Train:
  Rows with all 17 augmented features missing: 195
Validation:
  Rows with all 17 augmented features missing: 222
Test:
  Rows with all 17 augmented features missing: 102


In [27]:
# ============================================================
# CELL 22 — CREATE MISSINGNESS INDICATOR
# ============================================================

for data in [X_train, X_val, X_test]:
    data["augmented_features_missing"] = (
        data[augmented_features]
        .isna()
        .all(axis=1)
        .astype(int)
    )

print("Train missing indicator:")
print(X_train["augmented_features_missing"].value_counts())

print("\nValidation missing indicator:")
print(X_val["augmented_features_missing"].value_counts())

print("\nTest missing indicator:")
print(X_test["augmented_features_missing"].value_counts())

Train missing indicator:
augmented_features_missing
0    29699
1      195
Name: count, dtype: int64

Validation missing indicator:
augmented_features_missing
0    7607
1     222
Name: count, dtype: int64

Test missing indicator:
augmented_features_missing
0    8739
1     102
Name: count, dtype: int64


In [28]:
# ============================================================
# CELL 23 — VERIFY AUGMENTED MISSINGNESS PATTERN
# ============================================================

for name, data in [
    ("Train", X_train),
    ("Validation", X_val),
    ("Test", X_test)
]:
    
    partial_missing = (
        data[augmented_features].isna().sum(axis=1)
    )
    
    print(f"\n{name}")
    print("Rows with 0 missing augmented features :",
          (partial_missing == 0).sum())
    print("Rows with all 17 missing              :",
          (partial_missing == 17).sum())
    print("Rows with partial missingness          :",
          ((partial_missing > 0) & (partial_missing < 17)).sum())


Train
Rows with 0 missing augmented features : 29699
Rows with all 17 missing              : 195
Rows with partial missingness          : 0

Validation
Rows with 0 missing augmented features : 7607
Rows with all 17 missing              : 222
Rows with partial missingness          : 0

Test
Rows with 0 missing augmented features : 8739
Rows with all 17 missing              : 102
Rows with partial missingness          : 0


In [29]:
# ============================================================
# CELL 24 — CALCULATE TRAINING-SET MEDIANS
# ============================================================

# Calculate medians only from the training data
augmented_medians = X_train[augmented_features].median()

print("Training-set medians for augmented features:")

display(
    augmented_medians.to_frame("training_median")
)

Training-set medians for augmented features:


,training_median
in_txs_degree,1.000000
out_txs_degree,1.000000
total_BTC,0.405922
fees,0.000200
size,372.000000
num_input_addresses,2.000000
num_output_addresses,2.000000
in_BTC_min,0.033897
in_BTC_max,0.327172
in_BTC_mean,0.198910


In [30]:
# ============================================================
# CELL 25 — APPLY MEDIAN IMPUTATION
# ============================================================

# Apply training-derived medians to all three datasets
X_train[augmented_features] = X_train[augmented_features].fillna(
    augmented_medians
)

X_val[augmented_features] = X_val[augmented_features].fillna(
    augmented_medians
)

X_test[augmented_features] = X_test[augmented_features].fillna(
    augmented_medians
)

print("Missing values after imputation:")

print("Train:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

Missing values after imputation:
Train: 0
Validation: 0
Test: 0


In [31]:
# ============================================================
# CELL 26 — COMPARE FEATURE SCALES
# ============================================================

feature_scale_summary = pd.DataFrame({
    "Mean": X_train[model_feature_cols].mean(),
    "Std": X_train[model_feature_cols].std(),
    "Min": X_train[model_feature_cols].min(),
    "Median": X_train[model_feature_cols].median(),
    "Max": X_train[model_feature_cols].max()
})

display(feature_scale_summary)

,Mean,Std,Min,Median,Max
Time step,15.256941,10.278462,1.000000,15.000000,34.000000
Local_feature_1,-0.081301,0.554664,-0.172982,-0.170459,21.463992
Local_feature_2,-0.070584,0.549694,-0.210553,-0.158783,25.674475
Local_feature_3,-0.237289,1.018211,-1.756361,-0.646376,2.128587
Local_feature_4,0.149141,1.584176,-0.121970,-0.046932,49.027598
...,...,...,...,...,...
out_BTC_min,1.046537,15.065369,0.000000,0.014979,2010.466695
out_BTC_max,13.387567,83.094670,0.000054,0.329113,3174.493991
out_BTC_mean,5.179150,24.427543,0.000025,0.190470,2010.466695
out_BTC_median,4.609670,23.795776,0.000012,0.138419,2010.466695


In [32]:
# ============================================================
# CELL 27 — IDENTIFY LARGE-SCALE FEATURES
# ============================================================

scale_diagnostic = feature_scale_summary.copy()

scale_diagnostic["Abs_Max"] = scale_diagnostic["Max"].abs().combine(
    scale_diagnostic["Min"].abs(),
    max
)

scale_diagnostic["Max_to_Std"] = (
    scale_diagnostic["Abs_Max"] /
    scale_diagnostic["Std"].replace(0, np.nan)
)

display(
    scale_diagnostic
    .sort_values("Max_to_Std", ascending=False)
    .head(25)
)

,Mean,Std,Min,Median,Max,Abs_Max,Max_to_Std
Local_feature_15,-0.013148,0.009014,-0.013282,-0.013282,1.372120,1.372120,152.228538
out_BTC_min,1.046537,15.065369,0.000000,0.014979,2010.466695,2010.466695,133.449550
Local_feature_16,0.038272,1.373809,-0.057401,-0.056058,183.257462,183.257462,133.393680
out_txs_degree,1.105606,3.629339,0.000000,1.000000,472.000000,472.000000,130.051235
num_output_addresses,6.365525,104.906348,1.000000,2.000000,13107.000000,13107.000000,124.940008
Local_feature_14,0.045374,2.123794,-0.063200,-0.042955,265.263236,265.263236,124.900640
Local_feature_5,0.042753,2.082395,-0.063725,-0.043875,260.090707,260.090707,124.899793
Aggregate_feature_68,0.035949,2.046964,-0.125939,-0.087490,251.849028,251.849028,123.035391
Aggregate_feature_69,0.017615,1.996457,-0.131155,-0.131155,238.783493,238.783493,119.603645
Local_feature_36,0.029874,1.151779,-0.023045,-0.023045,129.089084,129.089084,112.077985


In [33]:
# ============================================================
# CELL 28 — COMPARE INHERITED AND AUGMENTED FEATURES
# ============================================================

inherited_features = (
    ["Time step"] +
    [f"Local_feature_{i}" for i in range(1, 94)] +
    [f"Aggregate_feature_{i}" for i in range(1, 73)]
)

augmented_features = [
    "in_txs_degree",
    "out_txs_degree",
    "total_BTC",
    "fees",
    "size",
    "num_input_addresses",
    "num_output_addresses",
    "in_BTC_min",
    "in_BTC_max",
    "in_BTC_mean",
    "in_BTC_median",
    "in_BTC_total",
    "out_BTC_min",
    "out_BTC_max",
    "out_BTC_mean",
    "out_BTC_median",
    "out_BTC_total"
]

scale_groups = pd.DataFrame({
    "Group": (
        ["Inherited"] * len(inherited_features) +
        ["Elliptic++ augmented"] * len(augmented_features)
    ),
    "Feature": inherited_features + augmented_features
})

scale_groups["Mean"] = scale_groups["Feature"].map(
    X_train.mean()
)

scale_groups["Std"] = scale_groups["Feature"].map(
    X_train.std()
)

display(
    scale_groups.groupby("Group")[["Mean", "Std"]]
    .agg(["mean", "median", "min", "max"])
)

Mean                                         Std  \
                           mean    median       min         max        mean   
Group                                                                         
Elliptic++ augmented  65.039526  9.340879  0.000540  980.800662  334.396581   
Inherited              0.183227  0.021432 -0.585123   15.256941    1.370301   

                                                        
                         median       min          max  
Group                                                   
Elliptic++ augmented  73.932679  0.002123  4834.813653  
Inherited              1.393510  0.009014    10.278462

In [34]:
# ============================================================
# CELL 29 — IDENTIFY APPROXIMATELY STANDARDIZED FEATURES
# ============================================================

feature_scale_check = pd.DataFrame({
    "Mean": X_train[inherited_features].mean(),
    "Std": X_train[inherited_features].std()
})

approximately_standardized = (
    feature_scale_check["Mean"].abs() < 0.1
) & (
    feature_scale_check["Std"].between(0.8, 1.2)
)

print(
    "Inherited features approximately standardized:",
    approximately_standardized.sum(),
    "out of",
    len(inherited_features)
)

print(
    "Inherited features not approximately standardized:",
    (~approximately_standardized).sum()
)

print("\nExamples of features not approximately standardized:")
display(
    feature_scale_check[~approximately_standardized]
    .head(20)
)

Inherited features approximately standardized: 28 out of 166
Inherited features not approximately standardized: 138

Examples of features not approximately standardized:


,Mean,Std
Time step,15.256941,10.278462
Local_feature_1,-0.081301,0.554664
Local_feature_2,-0.070584,0.549694
Local_feature_3,-0.237289,1.018211
Local_feature_4,0.149141,1.584176
Local_feature_5,0.042753,2.082395
Local_feature_6,0.157080,1.569308
Local_feature_7,-0.011197,0.610270
Local_feature_8,-0.103235,0.468787
Local_feature_9,-0.086280,0.530061


In [35]:
# Final modelling features
# Includes the original 183 transaction features
# plus the missingness indicator created earlier.

final_feature_cols = model_feature_cols + ["augmented_features_missing"]

print("Original modelling features :", len(model_feature_cols))
print("Missingness indicator       : 1")
print("Final feature count         :", len(final_feature_cols))

print("\nFinal training shape   :", X_train[final_feature_cols].shape)
print("Final validation shape :", X_val[final_feature_cols].shape)
print("Final test shape       :", X_test[final_feature_cols].shape)

Original modelling features : 183
Missingness indicator       : 1
Final feature count         : 184

Final training shape   : (29894, 184)
Final validation shape : (7829, 184)
Final test shape       : (8841, 184)


In [36]:
# Verify that the final modelling datasets are ready

print("Missing values:")
print("Train      :", X_train[final_feature_cols].isna().sum().sum())
print("Validation :", X_val[final_feature_cols].isna().sum().sum())
print("Test       :", X_test[final_feature_cols].isna().sum().sum())

print("\nInfinite values:")
print("Train      :", np.isinf(X_train[final_feature_cols]).sum().sum())
print("Validation :", np.isinf(X_val[final_feature_cols]).sum().sum())
print("Test       :", np.isinf(X_test[final_feature_cols]).sum().sum())

print("\nShapes:")
print("X_train :", X_train[final_feature_cols].shape)
print("X_val   :", X_val[final_feature_cols].shape)
print("X_test  :", X_test[final_feature_cols].shape)

Missing values:
Train      : 0
Validation : 0
Test       : 0

Infinite values:
Train      : 0
Validation : 0
Test       : 0

Shapes:
X_train : (29894, 184)
X_val   : (7829, 184)
X_test  : (8841, 184)


scaling

In [37]:
# Features that will be scaled
# The missingness indicator is binary, so we keep it unchanged.

scaling_features = model_feature_cols.copy()

print("Features to scale :", len(scaling_features))
print("Features not scaled :", ["augmented_features_missing"])

Features to scale : 183
Features not scaled : ['augmented_features_missing']


In [38]:
# Fit scaler ONLY on the training data
# This prevents validation/test information from entering preprocessing.

robust_scaler = RobustScaler()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scaling_features] = robust_scaler.fit_transform(
    X_train[scaling_features]
)

X_val_scaled[scaling_features] = robust_scaler.transform(
    X_val[scaling_features]
)

X_test_scaled[scaling_features] = robust_scaler.transform(
    X_test[scaling_features]
)

print("Robust scaling completed.")
print("Training shape   :", X_train_scaled.shape)
print("Validation shape :", X_val_scaled.shape)
print("Test shape       :", X_test_scaled.shape)

Robust scaling completed.
Training shape   : (29894, 184)
Validation shape : (7829, 184)
Test shape       : (8841, 184)


In [39]:
# Verify scaling behaviour on the training data

scaled_summary = pd.DataFrame({
    "Median": X_train_scaled[scaling_features].median(),
    "IQR": (
        X_train_scaled[scaling_features].quantile(0.75)
        - X_train_scaled[scaling_features].quantile(0.25)
    )
})

print("Median statistics after RobustScaler:")
print("Median of feature medians :", scaled_summary["Median"].median())
print("Maximum absolute median  :", scaled_summary["Median"].abs().max())

print("\nIQR statistics after RobustScaler:")
print("Median IQR :", scaled_summary["IQR"].median())
print("Minimum IQR:", scaled_summary["IQR"].min())
print("Maximum IQR:", scaled_summary["IQR"].max())

Median statistics after RobustScaler:
Median of feature medians : 0.0
Maximum absolute median  : 1.2784146343839887e-13

IQR statistics after RobustScaler:
Median IQR : 1.0
Minimum IQR: 0.0
Maximum IQR: 1.0000000000000637


In [40]:
# The missingness indicator should remain binary after scaling.

print("Unique values in missingness indicator:")

print(
    "Train      :",
    sorted(X_train_scaled["augmented_features_missing"].unique())
)

print(
    "Validation :",
    sorted(X_val_scaled["augmented_features_missing"].unique())
)

print(
    "Test       :",
    sorted(X_test_scaled["augmented_features_missing"].unique())
)

print("\nIndicator counts:")
print(
    X_train_scaled["augmented_features_missing"].value_counts().sort_index()
)

Unique values in missingness indicator:
Train      : [np.int64(0), np.int64(1)]
Validation : [np.int64(0), np.int64(1)]
Test       : [np.int64(0), np.int64(1)]

Indicator counts:
augmented_features_missing
0    29699
1      195
Name: count, dtype: int64


In [41]:
from sklearn.linear_model import LogisticRegression

# Use the RobustScaler version for Logistic Regression.
# Class weights compensate for the imbalanced illicit class.

X_train_lr = X_train_scaled[final_feature_cols]
X_val_lr = X_val_scaled[final_feature_cols]
X_test_lr = X_test_scaled[final_feature_cols]

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

print("Logistic Regression model prepared.")
print("Training data :", X_train_lr.shape)
print("Validation data :", X_val_lr.shape)
print("Test data :", X_test_lr.shape)

Logistic Regression model prepared.
Training data : (29894, 184)
Validation data : (7829, 184)
Test data : (8841, 184)


In [42]:
# Train ONLY on the training period.
# Validation and test data remain untouched.

lr_model.fit(X_train_lr, y_train)

print("Logistic Regression training completed.")
print("Number of coefficients :", len(lr_model.coef_[0]))
print("Intercept :", lr_model.intercept_[0])

Logistic Regression training completed.
Number of coefficients : 184
Intercept : 0.009547107864204917


In [43]:
# Generate predicted probabilities and class predictions

y_train_prob = lr_model.predict_proba(X_train_lr)[:, 1]
y_val_prob = lr_model.predict_proba(X_val_lr)[:, 1]
y_test_prob = lr_model.predict_proba(X_test_lr)[:, 1]

y_train_pred = (y_train_prob >= 0.5).astype(int)
y_val_pred = (y_val_prob >= 0.5).astype(int)
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("Predictions generated successfully.")

print("\nPrediction shapes:")
print("Train :", y_train_pred.shape)
print("Val   :", y_val_pred.shape)
print("Test  :", y_test_pred.shape)

Predictions generated successfully.

Prediction shapes:
Train : (29894,)
Val   : (7829,)
Test  : (8841,)


In [44]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(y_true, y_pred, y_prob, dataset_name):
    print(f"\n===== {dataset_name} =====")
    print("Precision :", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall    :", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1-score  :", round(f1_score(y_true, y_pred, zero_division=0), 4))
    print("ROC-AUC   :", round(roc_auc_score(y_true, y_prob), 4))
    print("PR-AUC    :", round(average_precision_score(y_true, y_prob), 4))

evaluate_model(y_train, y_train_pred, y_train_prob, "TRAIN")
evaluate_model(y_val, y_val_pred, y_val_prob, "VALIDATION")
evaluate_model(y_test, y_test_pred, y_test_prob, "TEST")


===== TRAIN =====
Precision : 0.5077
Recall    : 0.8798
F1-score  : 0.6438
ROC-AUC   : 0.9477
PR-AUC    : 0.6324

===== VALIDATION =====
Precision : 0.3758
Recall    : 0.8267
F1-score  : 0.5167
ROC-AUC   : 0.9321
PR-AUC    : 0.5435

===== TEST =====
Precision : 0.1939
Recall    : 0.6985
F1-score  : 0.3035
ROC-AUC   : 0.8921
PR-AUC    : 0.2889


In [45]:
from sklearn.metrics import confusion_matrix

train_cm = confusion_matrix(y_train, y_train_pred)
val_cm = confusion_matrix(y_val, y_val_pred)
test_cm = confusion_matrix(y_test, y_test_pred)

print("TRAIN confusion matrix:")
print(train_cm)

print("\nVALIDATION confusion matrix:")
print(val_cm)

print("\nTEST confusion matrix:")
print(test_cm)

TRAIN confusion matrix:
[[23478  2954]
 [  416  3046]]

VALIDATION confusion matrix:
[[6227  927]
 [ 117  558]]

TEST confusion matrix:
[[7248 1185]
 [ 123  285]]


In [46]:
def print_confusion_details(cm, dataset_name):
    tn, fp, fn, tp = cm.ravel()

    print(f"\n===== {dataset_name} =====")
    print("True Negatives  :", tn)
    print("False Positives :", fp)
    print("False Negatives :", fn)
    print("True Positives   :", tp)

    print("\nTotal transactions:", tn + fp + fn + tp)

print_confusion_details(train_cm, "TRAIN")
print_confusion_details(val_cm, "VALIDATION")
print_confusion_details(test_cm, "TEST")


===== TRAIN =====
True Negatives  : 23478
False Positives : 2954
False Negatives : 416
True Positives   : 3046

Total transactions: 29894

===== VALIDATION =====
True Negatives  : 6227
False Positives : 927
False Negatives : 117
True Positives   : 558

Total transactions: 7829

===== TEST =====
True Negatives  : 7248
False Positives : 1185
False Negatives : 123
True Positives   : 285

Total transactions: 8841


In [47]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:
    pred = (y_val_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.round(4)
)

,Threshold,Precision,Recall,F1
0,0.1,0.1892,0.9867,0.3174
1,0.2,0.2189,0.9733,0.3574
2,0.3,0.2539,0.9585,0.4015
3,0.4,0.2973,0.9052,0.4476
4,0.5,0.3758,0.8267,0.5167
5,0.6,0.5060,0.6844,0.5819
6,0.7,0.6076,0.5896,0.5985
7,0.8,0.6356,0.4652,0.5372
8,0.9,0.6461,0.3407,0.4462


In [48]:
best_row = threshold_df.loc[threshold_df["F1"].idxmax()]

best_threshold = best_row["Threshold"]

print("Best validation threshold based on F1:")
print("Threshold :", best_threshold)
print("Precision :", round(best_row["Precision"], 4))
print("Recall    :", round(best_row["Recall"], 4))
print("F1-score  :", round(best_row["F1"], 4))

Best validation threshold based on F1:
Threshold : 0.7
Precision : 0.6076
Recall    : 0.5896
F1-score  : 0.5985


In [53]:
# Apply the threshold selected using the validation set
# to the completely unseen test period.

selected_threshold = best_threshold

y_test_pred_tuned = (
    y_test_prob >= selected_threshold
).astype(int)

print("Selected threshold :", selected_threshold)

evaluate_model(
    y_test,
    y_test_pred_tuned,
    y_test_prob,
    "TEST - TUNED THRESHOLD"
)

Selected threshold : 0.7

===== TEST - TUNED THRESHOLD =====
Precision : 0.4049
Recall    : 0.4485
F1-score  : 0.4256
ROC-AUC   : 0.8921
PR-AUC    : 0.2889


In [54]:
test_tuned_cm = confusion_matrix(
    y_test,
    y_test_pred_tuned
)

print("TEST confusion matrix at threshold", selected_threshold)
print(test_tuned_cm)

print("\nDetailed results:")
print_confusion_details(
    test_tuned_cm,
    "TEST - TUNED THRESHOLD"
)

TEST confusion matrix at threshold 0.7
[[8164  269]
 [ 225  183]]

Detailed results:

===== TEST - TUNED THRESHOLD =====
True Negatives  : 8164
False Positives : 269
False Negatives : 225
True Positives   : 183

Total transactions: 8841


In [55]:
baseline_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Threshold": 0.50,
        "Precision": precision_score(y_test, y_test_pred, zero_division=0),
        "Recall": recall_score(y_test, y_test_pred, zero_division=0),
        "F1": f1_score(y_test, y_test_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_test_prob),
        "PR_AUC": average_precision_score(y_test, y_test_prob)
    },
    {
        "Model": "Logistic Regression",
        "Threshold": float(selected_threshold),
        "Precision": precision_score(y_test, y_test_pred_tuned, zero_division=0),
        "Recall": recall_score(y_test, y_test_pred_tuned, zero_division=0),
        "F1": f1_score(y_test, y_test_pred_tuned, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_test_prob),
        "PR_AUC": average_precision_score(y_test, y_test_prob)
    }
])

display(baseline_results.round(4))

,Model,Threshold,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression,0.5,0.1939,0.6985,0.3035,0.8921,0.2889
1,Logistic Regression,0.7,0.4049,0.4485,0.4256,0.8921,0.2889


In [56]:
baseline_results.to_csv(
    "logistic_regression_baseline_results.csv",
    index=False
)

print("Baseline results saved successfully.")

Baseline results saved successfully.


In [57]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest does not require feature scaling.
# Use the cleaned, imputed feature values.

X_train_rf = X_train[final_feature_cols]
X_val_rf = X_val[final_feature_cols]
X_test_rf = X_test[final_feature_cols]

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Random Forest model prepared.")
print("Training data   :", X_train_rf.shape)
print("Validation data :", X_val_rf.shape)
print("Test data       :", X_test_rf.shape)

Random Forest model prepared.
Training data   : (29894, 184)
Validation data : (7829, 184)
Test data       : (8841, 184)


In [58]:
# Train only on the training period.

rf_model.fit(X_train_rf, y_train)

print("Random Forest training completed.")
print("Number of trees :", rf_model.n_estimators)
print("Number of features used :", rf_model.n_features_in_)

Random Forest training completed.
Number of trees : 300
Number of features used : 184


In [59]:
# Generate probability predictions and class predictions

y_train_rf_prob = rf_model.predict_proba(X_train_rf)[:, 1]
y_val_rf_prob = rf_model.predict_proba(X_val_rf)[:, 1]
y_test_rf_prob = rf_model.predict_proba(X_test_rf)[:, 1]

y_train_rf_pred = (y_train_rf_prob >= 0.5).astype(int)
y_val_rf_pred = (y_val_rf_prob >= 0.5).astype(int)
y_test_rf_pred = (y_test_rf_prob >= 0.5).astype(int)

print("Random Forest predictions generated.")

print("\nPrediction shapes:")
print("Train :", y_train_rf_pred.shape)
print("Val   :", y_val_rf_pred.shape)
print("Test  :", y_test_rf_pred.shape)

Random Forest predictions generated.

Prediction shapes:
Train : (29894,)
Val   : (7829,)
Test  : (8841,)


In [60]:
evaluate_model(
    y_train,
    y_train_rf_pred,
    y_train_rf_prob,
    "RANDOM FOREST - TRAIN"
)

evaluate_model(
    y_val,
    y_val_rf_pred,
    y_val_rf_prob,
    "RANDOM FOREST - VALIDATION"
)

evaluate_model(
    y_test,
    y_test_rf_pred,
    y_test_rf_prob,
    "RANDOM FOREST - TEST"
)


===== RANDOM FOREST - TRAIN =====
Precision : 0.9994
Recall    : 1.0
F1-score  : 0.9997
ROC-AUC   : 1.0
PR-AUC    : 1.0

===== RANDOM FOREST - VALIDATION =====
Precision : 0.9575
Recall    : 0.8681
F1-score  : 0.9106
ROC-AUC   : 0.9798
PR-AUC    : 0.9302

===== RANDOM FOREST - TEST =====
Precision : 0.8465
Recall    : 0.473
F1-score  : 0.6069
ROC-AUC   : 0.853
PR-AUC    : 0.5469


In [61]:
rf_test_cm = confusion_matrix(
    y_test,
    y_test_rf_pred
)

print("Random Forest - TEST confusion matrix:")
print(rf_test_cm)

print("\nDetailed results:")
print_confusion_details(
    rf_test_cm,
    "RANDOM FOREST - TEST"
)

Random Forest - TEST confusion matrix:
[[8398   35]
 [ 215  193]]

Detailed results:

===== RANDOM FOREST - TEST =====
True Negatives  : 8398
False Positives : 35
False Negatives : 215
True Positives   : 193

Total transactions: 8841


In [62]:
# Random Forest feature importance
# This gives us an initial indication of which behavioural
# features are most influential in the model.

rf_importance = pd.DataFrame({
    "Feature": final_feature_cols,
    "Importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("Top 20 Random Forest features:")
display(rf_importance.head(20))

Top 20 Random Forest features:


,Feature,Importance
0,Local_feature_53,0.060273
1,Local_feature_41,0.047839
2,Local_feature_55,0.044676
3,size,0.044566
4,Local_feature_43,0.039193
5,Local_feature_49,0.037444
6,Local_feature_47,0.033141
7,Local_feature_5,0.024772
8,num_output_addresses,0.023647
9,Local_feature_90,0.023141


In [63]:
rf_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

rf_threshold_results = []

for threshold in rf_thresholds:
    pred = (y_val_rf_prob >= threshold).astype(int)

    rf_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, pred, zero_division=0
        )
    })

rf_threshold_df = pd.DataFrame(rf_threshold_results)

display(rf_threshold_df.round(4))

,Threshold,Precision,Recall,F1
0,0.1,0.3354,0.9570,0.4967
1,0.2,0.5222,0.9067,0.6627
2,0.3,0.7556,0.8978,0.8206
3,0.4,0.8835,0.8874,0.8854
4,0.5,0.9575,0.8681,0.9106
5,0.6,0.9930,0.8385,0.9092
6,0.7,0.9979,0.6904,0.8161
7,0.8,1.0000,0.6341,0.7761
8,0.9,1.0000,0.5467,0.7069


In [64]:
rf_best_row = rf_threshold_df.loc[
    rf_threshold_df["F1"].idxmax()
]

rf_best_threshold = rf_best_row["Threshold"]

print("Best Random Forest validation threshold:")
print("Threshold :", rf_best_threshold)
print("Precision :", round(rf_best_row["Precision"], 4))
print("Recall    :", round(rf_best_row["Recall"], 4))
print("F1-score  :", round(rf_best_row["F1"], 4))

Best Random Forest validation threshold:
Threshold : 0.5
Precision : 0.9575
Recall    : 0.8681
F1-score  : 0.9106


In [65]:
rf_test_metrics = {
    "Model": "Random Forest",
    "Threshold": rf_best_threshold,
    "Precision": precision_score(y_test, y_test_rf_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_rf_pred, zero_division=0),
    "F1": f1_score(y_test, y_test_rf_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, y_test_rf_prob),
    "PR_AUC": average_precision_score(y_test, y_test_rf_prob)
}

rf_results_df = pd.DataFrame([rf_test_metrics])

display(rf_results_df.round(4))

rf_results_df.to_csv(
    "random_forest_results.csv",
    index=False
)

print("Random Forest results saved successfully.")

,Model,Threshold,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Random Forest,0.5,0.8465,0.473,0.6069,0.853,0.5469


Random Forest results saved successfully.


In [66]:
model_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Precision": 0.4049,
        "Recall": 0.4485,
        "F1": 0.4256,
        "ROC_AUC": 0.8921,
        "PR_AUC": 0.2889
    },
    {
        "Model": "Random Forest",
        "Precision": rf_results_df.loc[0, "Precision"],
        "Recall": rf_results_df.loc[0, "Recall"],
        "F1": rf_results_df.loc[0, "F1"],
        "ROC_AUC": rf_results_df.loc[0, "ROC_AUC"],
        "PR_AUC": rf_results_df.loc[0, "PR_AUC"]
    }
])

display(model_comparison.round(4))

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression,0.4049,0.4485,0.4256,0.8921,0.2889
1,Random Forest,0.8465,0.4730,0.6069,0.8530,0.5469


In [67]:
rf_final_test_pred = (
    y_test_rf_prob >= rf_best_threshold
).astype(int)

rf_final_precision = precision_score(
    y_test, rf_final_test_pred, zero_division=0
)
rf_final_recall = recall_score(
    y_test, rf_final_test_pred, zero_division=0
)
rf_final_f1 = f1_score(
    y_test, rf_final_test_pred, zero_division=0
)

rf_final_cm = confusion_matrix(
    y_test, rf_final_test_pred
)

print("===== RANDOM FOREST - FINAL TEST =====")
print("Selected threshold :", rf_best_threshold)
print("Precision           :", round(rf_final_precision, 4))
print("Recall              :", round(rf_final_recall, 4))
print("F1-score            :", round(rf_final_f1, 4))
print("ROC-AUC             :", round(roc_auc_score(y_test, y_test_rf_prob), 4))
print("PR-AUC              :", round(average_precision_score(y_test, y_test_rf_prob), 4))

print("\nConfusion Matrix:")
print(rf_final_cm)

===== RANDOM FOREST - FINAL TEST =====
Selected threshold : 0.5
Precision           : 0.8465
Recall              : 0.473
F1-score            : 0.6069
ROC-AUC             : 0.853
PR-AUC              : 0.5469

Confusion Matrix:
[[8398   35]
 [ 215  193]]


In [68]:
final_model_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Precision": 0.4049,
        "Recall": 0.4485,
        "F1": 0.4256,
        "ROC-AUC": 0.8921,
        "PR-AUC": 0.2889
    },
    {
        "Model": "Random Forest",
        "Precision": rf_final_precision,
        "Recall": rf_final_recall,
        "F1": rf_final_f1,
        "ROC-AUC": roc_auc_score(y_test, y_test_rf_prob),
        "PR-AUC": average_precision_score(y_test, y_test_rf_prob)
    }
])

display(final_model_comparison.round(4))

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.4049,0.4485,0.4256,0.8921,0.2889
1,Random Forest,0.8465,0.4730,0.6069,0.8530,0.5469


In [69]:
feature_mapping = {
    "Local_feature_5": "Number of transaction outputs",
    "Local_feature_41": "Minimum incoming transactions of input addresses",
    "Local_feature_43": "Standard deviation of incoming transactions of input addresses",
    "Local_feature_47": "Minimum outgoing transactions of input addresses",
    "Local_feature_49": "Standard deviation of outgoing transactions of input addresses",
    "Local_feature_53": "Minimum lifetime of input addresses",
    "Local_feature_55": "Standard deviation of lifetime of input addresses",
    "Local_feature_65": "Minimum BTC sent by output addresses",
    "Local_feature_90": "Mean lifetime of output addresses",
    "Local_feature_14": "Spearman correlation of input values with indices",
    "Local_feature_18": "Maximum output value",
    "Local_feature_23": "Minimum received satoshi of input addresses",
    "Local_feature_25": "Standard deviation of received satoshi of input addresses",
    "Local_feature_52": "Spearman correlation of outgoing transactions of input addresses",
    "Local_feature_61": "Pearson correlation of received satoshi of output addresses",
    "Aggregate_feature_39": "Unspecified in recovered feature dictionary",
    "size": "Transaction size",
    "num_output_addresses": "Number of output addresses",
    "fees": "Transaction fees"
}

rf_top20_interpreted = rf_importance.head(20).copy()

rf_top20_interpreted["Meaning"] = (
    rf_top20_interpreted["Feature"]
    .map(feature_mapping)
    .fillna("Meaning not mapped yet")
)

display(rf_top20_interpreted)

,Feature,Importance,Meaning
0,Local_feature_53,0.060273,Minimum lifetime of input addresses
1,Local_feature_41,0.047839,Minimum incoming transactions of input addresses
2,Local_feature_55,0.044676,Standard deviation of lifetime of input addresses
3,size,0.044566,Transaction size
4,Local_feature_43,0.039193,Standard deviation of incoming transactions of...
5,Local_feature_49,0.037444,Standard deviation of outgoing transactions of...
6,Local_feature_47,0.033141,Minimum outgoing transactions of input addresses
7,Local_feature_5,0.024772,Number of transaction outputs
8,num_output_addresses,0.023647,Number of output addresses
9,Local_feature_90,0.023141,Mean lifetime of output addresses


In [70]:
def feature_group(feature):
    if feature.startswith("Local_feature_"):
        return "Local behavioural"
    elif feature.startswith("Aggregate_feature_"):
        return "Aggregate behavioural"
    elif feature in [
        "in_txs_degree",
        "out_txs_degree"
    ]:
        return "Transaction graph"
    elif feature in [
        "total_BTC",
        "fees",
        "size",
        "num_input_addresses",
        "num_output_addresses",
        "in_BTC_min",
        "in_BTC_max",
        "in_BTC_mean",
        "in_BTC_median",
        "in_BTC_total",
        "out_BTC_min",
        "out_BTC_max",
        "out_BTC_mean",
        "out_BTC_median",
        "out_BTC_total"
    ]:
        return "Augmented financial/structural"
    elif feature == "Time step":
        return "Temporal"
    elif feature == "augmented_features_missing":
        return "Missingness"
    else:
        return "Other"


rf_top20_interpreted["Behavioural_Group"] = (
    rf_top20_interpreted["Feature"].apply(feature_group)
)

display(
    rf_top20_interpreted[
        ["Feature", "Importance", "Behavioural_Group", "Meaning"]
    ]
)

,Feature,Importance,Behavioural_Group,Meaning
0,Local_feature_53,0.060273,Local behavioural,Minimum lifetime of input addresses
1,Local_feature_41,0.047839,Local behavioural,Minimum incoming transactions of input addresses
2,Local_feature_55,0.044676,Local behavioural,Standard deviation of lifetime of input addresses
3,size,0.044566,Augmented financial/structural,Transaction size
4,Local_feature_43,0.039193,Local behavioural,Standard deviation of incoming transactions of...
5,Local_feature_49,0.037444,Local behavioural,Standard deviation of outgoing transactions of...
6,Local_feature_47,0.033141,Local behavioural,Minimum outgoing transactions of input addresses
7,Local_feature_5,0.024772,Local behavioural,Number of transaction outputs
8,num_output_addresses,0.023647,Augmented financial/structural,Number of output addresses
9,Local_feature_90,0.023141,Local behavioural,Mean lifetime of output addresses


In [71]:
# Features that may potentially contain information derived from
# transaction/address history or neighbouring transactions.

temporal_risk_features = []

for col in model_feature_cols:
    if (
        col.startswith("Local_feature_")
        or col.startswith("Aggregate_feature_")
        or col in [
            "in_txs_degree",
            "out_txs_degree"
        ]
    ):
        temporal_risk_features.append(col)

print("Total model features:", len(model_feature_cols))
print("Potential temporal/graph-derived features:", len(temporal_risk_features))

print("\nFirst 20 candidates:")
print(temporal_risk_features[:20])

Total model features: 183
Potential temporal/graph-derived features: 167

First 20 candidates:
['Local_feature_1', 'Local_feature_2', 'Local_feature_3', 'Local_feature_4', 'Local_feature_5', 'Local_feature_6', 'Local_feature_7', 'Local_feature_8', 'Local_feature_9', 'Local_feature_10', 'Local_feature_11', 'Local_feature_12', 'Local_feature_13', 'Local_feature_14', 'Local_feature_15', 'Local_feature_16', 'Local_feature_17', 'Local_feature_18', 'Local_feature_19', 'Local_feature_20']


In [72]:
time_target_summary = (
    labelled_data
    .groupby("Time step")["target"]
    .agg(
        Transactions="count",
        Illicit="sum",
        Illicit_Rate="mean"
    )
    .reset_index()
)

time_target_summary["Illicit_Rate"] *= 100

display(time_target_summary.round(2))

,Time step,Transactions,Illicit,Illicit_Rate
0,1,2147,17,0.79
1,2,1117,18,1.61
2,3,1279,11,0.86
3,4,1440,30,2.08
4,5,1882,8,0.43
5,6,485,5,1.03
6,7,1203,102,8.48
7,8,1165,67,5.75
8,9,778,248,31.88
9,10,972,18,1.85


In [73]:
rf_importance["Is_Temporal_Graph_Derived"] = (
    rf_importance["Feature"].isin(temporal_risk_features)
)

importance_summary = (
    rf_importance
    .groupby("Is_Temporal_Graph_Derived")["Importance"]
    .agg(
        Feature_Count="count",
        Total_Importance="sum",
        Mean_Importance="mean"
    )
    .reset_index()
)

display(importance_summary.round(4))

,Is_Temporal_Graph_Derived,Feature_Count,Total_Importance,Mean_Importance
0,False,17,0.1264,0.0074
1,True,167,0.8736,0.0052


In [74]:
top20_audit = rf_importance.head(20).copy()

top20_audit["Temporal_Graph_Derived"] = (
    top20_audit["Feature"].isin(temporal_risk_features)
)

display(
    top20_audit[
        ["Feature", "Importance", "Temporal_Graph_Derived"]
    ].round(4)
)

print(
    "\nNumber of top-20 features that are "
    "temporal/graph-derived:",
    top20_audit["Temporal_Graph_Derived"].sum()
)

,Feature,Importance,Temporal_Graph_Derived
0,Local_feature_53,0.0603,True
1,Local_feature_41,0.0478,True
2,Local_feature_55,0.0447,True
3,size,0.0446,False
4,Local_feature_43,0.0392,True
5,Local_feature_49,0.0374,True
6,Local_feature_47,0.0331,True
7,Local_feature_5,0.0248,True
8,num_output_addresses,0.0236,False
9,Local_feature_90,0.0231,True



Number of top-20 features that are temporal/graph-derived: 17


In [75]:
split_feature_summary = pd.DataFrame({
    "Train_Mean": X_train[model_feature_cols].mean(),
    "Validation_Mean": X_val[model_feature_cols].mean(),
    "Test_Mean": X_test[model_feature_cols].mean(),
    "Train_Median": X_train[model_feature_cols].median(),
    "Validation_Median": X_val[model_feature_cols].median(),
    "Test_Median": X_test[model_feature_cols].median()
})

split_feature_summary["Test_vs_Train_Mean_Diff"] = (
    split_feature_summary["Test_Mean"]
    - split_feature_summary["Train_Mean"]
).abs()
    
display(
    split_feature_summary
    .sort_values("Test_vs_Train_Mean_Diff", ascending=False)
    .head(20)
)

,Train_Mean,Validation_Mean,Test_Mean,Train_Median,Validation_Median,Test_Median,Test_vs_Train_Mean_Diff
size,980.800662,1005.753097,1090.240244,372.000000,226.000000,361.000000,109.439582
Time step,15.256941,37.880445,44.426309,15.000000,38.000000,44.000000,29.169368
in_BTC_total,14.518172,35.846994,19.489474,0.406391,0.406391,0.285756,4.971302
out_BTC_total,14.517630,35.844354,19.486159,0.405922,0.405922,0.284439,4.968529
total_BTC,14.517630,35.844354,19.486159,0.405922,0.405922,0.284439,4.968529
in_BTC_max,13.054122,33.453967,17.082863,0.327172,0.327172,0.210260,4.028741
out_BTC_max,13.387567,33.821771,17.184257,0.329113,0.329113,0.223740,3.796690
in_BTC_min,9.340879,29.861466,12.099463,0.033897,0.085000,0.033771,2.758584
in_BTC_mean,10.539672,30.875655,13.036688,0.198910,0.198910,0.133356,2.497016
in_BTC_median,10.311384,30.636413,12.718021,0.135030,0.135030,0.079472,2.406637


In [76]:
feature_shift = pd.DataFrame({
    "Feature": model_feature_cols,
    "Train_Mean": X_train[model_feature_cols].mean().values,
    "Test_Mean": X_test[model_feature_cols].mean().values,
    "Train_Std": X_train[model_feature_cols].std().values,
    "Test_Std": X_test[model_feature_cols].std().values
})

feature_shift["Mean_Absolute_Difference"] = (
    feature_shift["Test_Mean"]
    - feature_shift["Train_Mean"]
).abs()

feature_shift["Std_Ratio"] = (
    feature_shift["Test_Std"]
    / feature_shift["Train_Std"].replace(0, np.nan)
)

display(
    feature_shift
    .sort_values("Mean_Absolute_Difference", ascending=False)
    .head(20)
)

,Feature,Train_Mean,Test_Mean,Train_Std,Test_Std,Mean_Absolute_Difference,Std_Ratio
170,size,980.800662,1090.240244,4834.813653,3639.140983,109.439582,0.752695
0,Time step,15.256941,44.426309,10.278462,2.113794,29.169368,0.205653
177,in_BTC_total,14.518172,19.489474,87.931188,123.327042,4.971302,1.402540
182,out_BTC_total,14.517630,19.486159,87.931072,123.326159,4.968529,1.402532
168,total_BTC,14.517630,19.486159,87.931072,123.326159,4.968529,1.402532
174,in_BTC_max,13.054122,17.082863,83.296299,119.249599,4.028741,1.431631
179,out_BTC_max,13.387567,17.184257,83.094670,116.624189,3.796690,1.403510
173,in_BTC_min,9.340879,12.099463,72.632716,90.067572,2.758584,1.240041
175,in_BTC_mean,10.539672,13.036688,73.986482,90.905176,2.497016,1.228673
176,in_BTC_median,10.311384,12.718021,73.932679,90.810828,2.406637,1.228291


In [77]:
augmented_check_cols = [
    "in_txs_degree",
    "out_txs_degree",
    "num_input_addresses",
    "num_output_addresses",
    "total_BTC",
    "fees",
    "size"
]

augmented_summary = pd.DataFrame({
    "Missing_Count": X[augmented_check_cols].isna().sum(),
    "Train_Mean": X_train[augmented_check_cols].mean(),
    "Test_Mean": X_test[augmented_check_cols].mean(),
    "Train_Median": X_train[augmented_check_cols].median(),
    "Test_Median": X_test[augmented_check_cols].median()
})

display(augmented_summary.round(4))

,Missing_Count,Train_Mean,Test_Mean,Train_Median,Test_Median
in_txs_degree,965,1.7621,2.0355,1.0000,1.0000
out_txs_degree,965,1.1056,1.2371,1.0000,1.0000
num_input_addresses,965,4.6151,5.5822,2.0000,1.0000
num_output_addresses,965,6.3655,4.8939,2.0000,2.0000
total_BTC,965,14.5176,19.4862,0.4059,0.2844
fees,965,0.0005,0.0033,0.0002,0.0009
size,965,980.8007,1090.2402,372.0000,361.0000


In [79]:
# Reconstruct basic graph degree information from the transaction edge list

tx_edges = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\txs_edgelist.csv")

in_degree_raw = tx_edges.groupby("txId2").size()
out_degree_raw = tx_edges.groupby("txId1").size()

degree_check = pd.DataFrame({
    "Stored_In_Degree": tx_features.set_index("txId")["in_txs_degree"],
    "Stored_Out_Degree": tx_features.set_index("txId")["out_txs_degree"],
    "Raw_In_Degree": in_degree_raw,
    "Raw_Out_Degree": out_degree_raw
})

degree_check["Raw_In_Degree"] = degree_check["Raw_In_Degree"].fillna(0)
degree_check["Raw_Out_Degree"] = degree_check["Raw_Out_Degree"].fillna(0)

degree_check["In_Degree_Match"] = (
    degree_check["Stored_In_Degree"] == degree_check["Raw_In_Degree"]
)

degree_check["Out_Degree_Match"] = (
    degree_check["Stored_Out_Degree"] == degree_check["Raw_Out_Degree"]
)

print(
    "Input-degree matches :",
    degree_check["In_Degree_Match"].mean()
)

print(
    "Output-degree matches:",
    degree_check["Out_Degree_Match"].mean()
)

display(degree_check.head())

Input-degree matches : 0.995264245297371
Output-degree matches: 0.995264245297371


,Stored_In_Degree,Stored_Out_Degree,Raw_In_Degree,Raw_Out_Degree,In_Degree_Match,Out_Degree_Match
1076,1.0,1.0,1.0,1.0,True,True
2534,0.0,3.0,0.0,3.0,True,True
3181,1.0,112.0,1.0,112.0,True,True
3321,1.0,0.0,1.0,0.0,True,True
3889,0.0,50.0,0.0,50.0,True,True


In [80]:
# Print all inherited feature names with their original feature numbers.
# This helps us audit which features depend on historical address behaviour.

inherited_features = (
    ["Time step"] +
    [f"Local_feature_{i}" for i in range(1, 94)] +
    [f"Aggregate_feature_{i}" for i in range(1, 73)]
)

print("Number of inherited features:", len(inherited_features))

for i, feature in enumerate(inherited_features, start=1):
    print(f"Original Feature {i:3d} -> {feature}")

Number of inherited features: 166
Original Feature   1 -> Time step
Original Feature   2 -> Local_feature_1
Original Feature   3 -> Local_feature_2
Original Feature   4 -> Local_feature_3
Original Feature   5 -> Local_feature_4
Original Feature   6 -> Local_feature_5
Original Feature   7 -> Local_feature_6
Original Feature   8 -> Local_feature_7
Original Feature   9 -> Local_feature_8
Original Feature  10 -> Local_feature_9
Original Feature  11 -> Local_feature_10
Original Feature  12 -> Local_feature_11
Original Feature  13 -> Local_feature_12
Original Feature  14 -> Local_feature_13
Original Feature  15 -> Local_feature_14
Original Feature  16 -> Local_feature_15
Original Feature  17 -> Local_feature_16
Original Feature  18 -> Local_feature_17
Original Feature  19 -> Local_feature_18
Original Feature  20 -> Local_feature_19
Original Feature  21 -> Local_feature_20
Original Feature  22 -> Local_feature_21
Original Feature  23 -> Local_feature_22
Original Feature  24 -> Local_feature_2

In [81]:
address_history_ranges = {
    "Input address history": list(range(23, 59)),
    "Output address history": list(range(59, 95)),
    "Aggregate transaction history": list(range(95, 167))
}

for group, feature_numbers in address_history_ranges.items():
    print(f"\n{group}")
    print("-" * len(group))

    for number in feature_numbers:
        if number == 1:
            feature_name = "Time step"
        elif 2 <= number <= 94:
            feature_name = f"Local_feature_{number - 1}"
        else:
            feature_name = f"Aggregate_feature_{number - 94}"

        print(f"{number:3d} -> {feature_name}")


Input address history
---------------------
 23 -> Local_feature_22
 24 -> Local_feature_23
 25 -> Local_feature_24
 26 -> Local_feature_25
 27 -> Local_feature_26
 28 -> Local_feature_27
 29 -> Local_feature_28
 30 -> Local_feature_29
 31 -> Local_feature_30
 32 -> Local_feature_31
 33 -> Local_feature_32
 34 -> Local_feature_33
 35 -> Local_feature_34
 36 -> Local_feature_35
 37 -> Local_feature_36
 38 -> Local_feature_37
 39 -> Local_feature_38
 40 -> Local_feature_39
 41 -> Local_feature_40
 42 -> Local_feature_41
 43 -> Local_feature_42
 44 -> Local_feature_43
 45 -> Local_feature_44
 46 -> Local_feature_45
 47 -> Local_feature_46
 48 -> Local_feature_47
 49 -> Local_feature_48
 50 -> Local_feature_49
 51 -> Local_feature_50
 52 -> Local_feature_51
 53 -> Local_feature_52
 54 -> Local_feature_53
 55 -> Local_feature_54
 56 -> Local_feature_55
 57 -> Local_feature_56
 58 -> Local_feature_57

Output address history
----------------------
 59 -> Local_feature_58
 60 -> Local_feature

In [82]:
# Explicit address-history features from the recovered Elliptic
# feature semantics.

input_address_history = [
    f"Local_feature_{i}" for i in range(23, 58)
]

output_address_history = [
    f"Local_feature_{i}" for i in range(59, 94)
]

address_history_features = (
    input_address_history +
    output_address_history
)

print("Input-address history features :", len(input_address_history))
print("Output-address history features:", len(output_address_history))
print("Total address-history features  :", len(address_history_features))

Input-address history features : 35
Output-address history features: 35
Total address-history features  : 70


In [83]:
address_importance = (
    rf_importance[
        rf_importance["Feature"].isin(address_history_features)
    ]
    .copy()
)

address_importance["Cumulative_Importance"] = (
    address_importance["Importance"].cumsum()
)

print(
    "Total RF importance from address-history features:",
    round(address_importance["Importance"].sum(), 4)
)

display(
    address_importance.head(20).round(4)
)

Total RF importance from address-history features: 0.5546


,Feature,Importance,Is_Temporal_Graph_Derived,Cumulative_Importance
0,Local_feature_53,0.0603,True,0.0603
1,Local_feature_41,0.0478,True,0.1081
2,Local_feature_55,0.0447,True,0.1528
4,Local_feature_43,0.0392,True,0.1920
5,Local_feature_49,0.0374,True,0.2294
6,Local_feature_47,0.0331,True,0.2626
9,Local_feature_90,0.0231,True,0.2857
13,Local_feature_65,0.0166,True,0.3023
15,Local_feature_23,0.0137,True,0.3160
17,Local_feature_52,0.0133,True,0.3293


In [84]:
top_address_features = address_importance.head(10)["Feature"].tolist()

print("Top address-history features:")
for feature in top_address_features:
    print("-", feature)

Top address-history features:
- Local_feature_53
- Local_feature_41
- Local_feature_55
- Local_feature_43
- Local_feature_49
- Local_feature_47
- Local_feature_90
- Local_feature_65
- Local_feature_23
- Local_feature_52


In [86]:
for feature in top_address_features:
    print(f"\n{feature}")
    print("-" * len(feature))
    
    temp = labelled_data[["Time step", feature]].copy()
    
    print(
        temp.groupby("Time step")[feature]
        .agg(["count", "mean", "median", "min", "max"])
        .head(10)
    )


Local_feature_53
----------------
           count      mean    median       min       max
Time step                                               
1           2147  0.934727  0.085341 -0.488362  4.920559
2           1117  1.155747  0.228950 -0.488358  4.361162
3           1279  0.947468  0.028873 -0.488358  4.263388
4           1440  0.868819 -0.139183 -0.488362  4.514022
5           1882  1.339151  0.560579 -0.488362  4.263388
6            485  0.842027  0.024540 -0.488362  4.799484
7           1203  1.527392  1.414935 -0.488360  4.208015
8           1165  1.700166  1.978764 -0.488362  4.920633
9            778  0.419549 -0.323436 -0.488358  4.197316
10           972  1.646664  1.503941 -0.488356  4.920559

Local_feature_41
----------------
           count      mean    median       min        max
Time step                                                
1           2147  0.452796 -0.237631 -0.239368  16.682848
2           1117  0.217348 -0.237293 -0.239368   8.860960
3           12

In [87]:
feature = "Local_feature_53"

temp = labelled_data[["Time step", feature]].copy()

# For each time step, calculate the maximum feature value
time_summary = (
    temp.groupby("Time step")[feature]
    .agg(["count", "min", "median", "mean", "max"])
)

display(time_summary)

,count,min,median,mean,max
Time step,,,,,
1,2147,-0.488362,0.085341,0.934727,4.920559
2,1117,-0.488358,0.228950,1.155747,4.361162
3,1279,-0.488358,0.028873,0.947468,4.263388
4,1440,-0.488362,-0.139183,0.868819,4.514022
5,1882,-0.488362,0.560579,1.339151,4.263388
6,485,-0.488362,0.024540,0.842027,4.799484
7,1203,-0.488360,1.414935,1.527392,4.208015
8,1165,-0.488362,1.978764,1.700166,4.920633
9,778,-0.488358,-0.323436,0.419549,4.197316


In [88]:
feature = "Local_feature_53"

# Find the most common exact values in the feature
value_counts = (
    labelled_data[feature]
    .round(6)
    .value_counts()
    .head(15)
)

print(f"Most frequent values for {feature}:")
display(value_counts)

Most frequent values for Local_feature_53:


Local_feature_53
 0.481198    1453
 2.479235     847
 2.905110     593
 0.611341     488
 1.673413     444
 3.702378     436
 3.698812     426
 1.008086     420
 3.701827     415
 3.701130     400
 3.693229     376
-0.488362     374
 3.696310     369
 3.693767     363
 3.698791     350
Name: count, dtype: int64

In [89]:
print("Transaction time-step information")
print("=" * 40)

print("Minimum time step:", tx_features["Time step"].min())
print("Maximum time step:", tx_features["Time step"].max())
print("Number of time steps:", tx_features["Time step"].nunique())

print("\nTransactions per time step:")
display(
    tx_features["Time step"]
    .value_counts()
    .sort_index()
    .to_frame("Transaction_Count")
)

Transaction time-step information
Minimum time step: 1
Maximum time step: 49
Number of time steps: 49

Transactions per time step:


,Transaction_Count
Time step,
1,7880
2,4544
3,6621
4,5693
5,6803
6,4328
7,6048
8,4457
9,4996


In [91]:
# Load the transaction graph
tx_edges = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\txs_edgelist.csv")

# Attach time steps to both ends of every edge
tx_time = tx_features[["txId", "Time step"]].copy()

tx_edges_timed = (
    tx_edges
    .merge(
        tx_time.rename(columns={"txId": "txId1", "Time step": "time1"}),
        on="txId1",
        how="left"
    )
    .merge(
        tx_time.rename(columns={"txId": "txId2", "Time step": "time2"}),
        on="txId2",
        how="left"
    )
)

# Classify edge timing
tx_edges_timed["temporal_relation"] = np.select(
    [
        tx_edges_timed["time2"] < tx_edges_timed["time1"],
        tx_edges_timed["time2"] == tx_edges_timed["time1"],
        tx_edges_timed["time2"] > tx_edges_timed["time1"]
    ],
    [
        "Past -> Current",
        "Same time step",
        "Current -> Future"
    ],
    default="Unknown"
)

print("Transaction edge temporal relationships:")
display(
    tx_edges_timed["temporal_relation"]
    .value_counts()
    .to_frame("Edge_Count")
)

print("\nPercentage:")
display(
    (
        tx_edges_timed["temporal_relation"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_frame("Percentage")
    )
)

Transaction edge temporal relationships:


,Edge_Count
temporal_relation,
Same time step,234355



Percentage:


,Percentage
temporal_relation,
Same time step,100.0


In [92]:
wallets_features = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\wallets_features.csv")

print("Wallet temporal data shape:", wallets_features.shape)
print("\nColumns:")
print(wallets_features.columns.tolist())

print("\nFirst 5 rows:")
display(wallets_features.head())

Wallet temporal data shape: (1268260, 57)

Columns:
['address', 'Time step', 'num_txs_as_sender', 'num_txs_as receiver', 'first_block_appeared_in', 'last_block_appeared_in', 'lifetime_in_blocks', 'total_txs', 'first_sent_block', 'first_received_block', 'num_timesteps_appeared_in', 'btc_transacted_total', 'btc_transacted_min', 'btc_transacted_max', 'btc_transacted_mean', 'btc_transacted_median', 'btc_sent_total', 'btc_sent_min', 'btc_sent_max', 'btc_sent_mean', 'btc_sent_median', 'btc_received_total', 'btc_received_min', 'btc_received_max', 'btc_received_mean', 'btc_received_median', 'fees_total', 'fees_min', 'fees_max', 'fees_mean', 'fees_median', 'fees_as_share_total', 'fees_as_share_min', 'fees_as_share_max', 'fees_as_share_mean', 'fees_as_share_median', 'blocks_btwn_txs_total', 'blocks_btwn_txs_min', 'blocks_btwn_txs_max', 'blocks_btwn_txs_mean', 'blocks_btwn_txs_median', 'blocks_btwn_input_txs_total', 'blocks_btwn_input_txs_min', 'blocks_btwn_input_txs_max', 'blocks_btwn_input_txs_

,address,Time step,num_txs_as_sender,num_txs_as receiver,first_block_appeared_in,last_block_appeared_in,lifetime_in_blocks,total_txs,first_sent_block,first_received_block,num_timesteps_appeared_in,btc_transacted_total,btc_transacted_min,btc_transacted_max,btc_transacted_mean,btc_transacted_median,btc_sent_total,btc_sent_min,btc_sent_max,btc_sent_mean,btc_sent_median,btc_received_total,btc_received_min,btc_received_max,btc_received_mean,btc_received_median,fees_total,fees_min,fees_max,fees_mean,fees_median,fees_as_share_total,fees_as_share_min,fees_as_share_max,fees_as_share_mean,fees_as_share_median,blocks_btwn_txs_total,blocks_btwn_txs_min,blocks_btwn_txs_max,blocks_btwn_txs_mean,blocks_btwn_txs_median,blocks_btwn_input_txs_total,blocks_btwn_input_txs_min,blocks_btwn_input_txs_max,blocks_btwn_input_txs_mean,blocks_btwn_input_txs_median,blocks_btwn_output_txs_total,blocks_btwn_output_txs_min,blocks_btwn_output_txs_max,blocks_btwn_output_txs_mean,blocks_btwn_output_txs_median,num_addr_transacted_multiple,transacted_w_address_total,transacted_w_address_min,transacted_w_address_max,transacted_w_address_mean,transacted_w_address_median
0,111112TykSw72ztDN2WJger4cynzWYC5w,25,0.0,1.0,439586.0,439586.0,0.0,1.0,0.0,439586.0,1.0,0.010628,0.010628,0.010628,0.010628,0.010628,0.0,0.0,0.0,0.0,0.0,0.010628,0.010628,0.010628,0.010628,0.010628,0.007042,0.007042,0.007042,0.007042,0.007042,0.000012,0.000012,0.000012,0.000012,0.000012,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,24.0,1.0,1.0,1.0,1.0
1,1111DAYXhoxZx2tsRnzimfozo783x1yC2,25,0.0,8.0,439589.0,485959.0,46370.0,8.0,0.0,439589.0,6.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.0,0.0,0.0,0.0,0.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.002371,0.000122,0.000580,0.000296,0.000242,0.002217,0.000121,0.000523,0.000277,0.000237,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,0.0,0.0,0.0,0.0,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,8.0,1.0,1.0,1.0,1.0
2,1111DAYXhoxZx2tsRnzimfozo783x1yC2,29,0.0,8.0,439589.0,485959.0,46370.0,8.0,0.0,439589.0,6.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.0,0.0,0.0,0.0,0.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.002371,0.000122,0.000580,0.000296,0.000242,0.002217,0.000121,0.000523,0.000277,0.000237,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,0.0,0.0,0.0,0.0,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,8.0,1.0,1.0,1.0,1.0
3,1111DAYXhoxZx2tsRnzimfozo783x1yC2,39,0.0,8.0,439589.0,485959.0,46370.0,8.0,0.0,439589.0,6.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.0,0.0,0.0,0.0,0.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.002371,0.000122,0.000580,0.000296,0.000242,0.002217,0.000121,0.000523,0.000277,0.000237,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,0.0,0.0,0.0,0.0,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,8.0,1.0,1.0,1.0,1.0
4,1111DAYXhoxZx2tsRnzimfozo783x1yC2,39,0.0,8.0,439589.0,485959.0,46370.0,8.0,0.0,439589.0,6.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.0,0.0,0.0,0.0,0.0,0.273046,0.003900,0.133777,0.034131,0.014352,0.002371,0.000122,0.000580,0.000296,0.000242,0.002217,0.000121,0.000523,0.000277,0.000237,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,0.0,0.0,0.0,0.0,46370.0,0.0,20164.0,6624.285714,8060.0,0.0,8.0,1.0,1.0,1.0,1.0


In [93]:
print("Wallet temporal coverage")
print("=" * 40)

print("Number of unique wallets:",
      wallets_features["address"].nunique())

print("Minimum time step:",
      wallets_features["Time step"].min())

print("Maximum time step:",
      wallets_features["Time step"].max())

print("Number of unique time steps:",
      wallets_features["Time step"].nunique())

print("\nWallet records per time step:")
display(
    wallets_features["Time step"]
    .value_counts()
    .sort_index()
    .to_frame("Wallet_Record_Count")
)

Wallet temporal coverage
Number of unique wallets: 822942
Minimum time step: 1
Maximum time step: 49
Number of unique time steps: 49

Wallet records per time step:


,Wallet_Record_Count
Time step,
1,51108
2,32269
3,31106
4,39948
5,35140
6,19003
7,31722
8,34127
9,28673


In [94]:
wallet_time_check = wallets_features[
    ["address", "Time step", "first_block_appeared_in",
     "last_block_appeared_in", "lifetime_in_blocks",
     "total_txs"]
].copy()

print("Rows where last appearance block is present:")
print(
    wallet_time_check["last_block_appeared_in"]
    .notna()
    .sum()
)

print("\nSample records where an address appears at multiple time steps:")

multi_addresses = (
    wallets_features.groupby("address")["Time step"]
    .nunique()
)

multi_addresses = multi_addresses[multi_addresses > 1].index

display(
    wallets_features[
        wallets_features["address"].isin(multi_addresses)
    ][
        ["address", "Time step",
         "first_block_appeared_in",
         "last_block_appeared_in",
         "lifetime_in_blocks",
         "total_txs"]
    ]
    .sort_values(["address", "Time step"])
    .head(30)
)

Rows where last appearance block is present:
1268260

Sample records where an address appears at multiple time steps:


,address,Time step,first_block_appeared_in,last_block_appeared_in,lifetime_in_blocks,total_txs
1,1111DAYXhoxZx2tsRnzimfozo783x1yC2,25,439589.0,485959.0,46370.0,8.0
2,1111DAYXhoxZx2tsRnzimfozo783x1yC2,29,439589.0,485959.0,46370.0,8.0
3,1111DAYXhoxZx2tsRnzimfozo783x1yC2,39,439589.0,485959.0,46370.0,8.0
4,1111DAYXhoxZx2tsRnzimfozo783x1yC2,39,439589.0,485959.0,46370.0,8.0
5,1111DAYXhoxZx2tsRnzimfozo783x1yC2,43,439589.0,485959.0,46370.0,8.0
6,1111DAYXhoxZx2tsRnzimfozo783x1yC2,43,439589.0,485959.0,46370.0,8.0
7,1111DAYXhoxZx2tsRnzimfozo783x1yC2,47,439589.0,485959.0,46370.0,8.0
8,1111DAYXhoxZx2tsRnzimfozo783x1yC2,48,439589.0,485959.0,46370.0,8.0
29,111khWGs3Mj7UgKT7aS6hLsYK11ii4Gmw,14,417408.0,457732.0,40324.0,5.0
30,111khWGs3Mj7UgKT7aS6hLsYK11ii4Gmw,34,417408.0,457732.0,40324.0,5.0


In [95]:
# For each wallet record, compare its last observed block
# with its first observed block and lifetime.

wallet_check = wallets_features[
    ["address", "Time step",
     "first_block_appeared_in",
     "last_block_appeared_in",
     "lifetime_in_blocks"]
].copy()

wallet_check["computed_lifetime"] = (
    wallet_check["last_block_appeared_in"]
    - wallet_check["first_block_appeared_in"]
)

wallet_check["lifetime_matches"] = np.isclose(
    wallet_check["lifetime_in_blocks"],
    wallet_check["computed_lifetime"],
    equal_nan=True
)

print("Lifetime consistency:")
print(
    wallet_check["lifetime_matches"]
    .value_counts()
)

print("\nPercentage matching:")
print(
    wallet_check["lifetime_matches"]
    .mean() * 100
)

Lifetime consistency:
lifetime_matches
True    1268260
Name: count, dtype: int64

Percentage matching:
100.0


In [96]:
# Check whether wallet-level historical fields remain unchanged
# when the same address appears at multiple time steps.

multi_wallets = wallets_features.groupby("address").filter(
    lambda g: g["Time step"].nunique() > 1
)

wallet_consistency = (
    multi_wallets
    .groupby("address")
    .agg(
        time_steps=("Time step", "nunique"),
        first_block_unique=("first_block_appeared_in", "nunique"),
        last_block_unique=("last_block_appeared_in", "nunique"),
        lifetime_unique=("lifetime_in_blocks", "nunique"),
        total_txs_unique=("total_txs", "nunique")
    )
)

print("Addresses appearing at multiple time steps:",
      len(wallet_consistency))

print("\nAddresses where historical fields remain constant:")

constant_history = (
    (wallet_consistency["first_block_unique"] == 1) &
    (wallet_consistency["last_block_unique"] == 1) &
    (wallet_consistency["lifetime_unique"] == 1) &
    (wallet_consistency["total_txs_unique"] == 1)
)

print("Count:", constant_history.sum())
print(
    "Percentage:",
    constant_history.mean() * 100
)

display(
    wallet_consistency
    .sort_values("time_steps", ascending=False)
    .head(20)
)

Addresses appearing at multiple time steps: 61487

Addresses where historical fields remain constant:
Count: 61487
Percentage: 100.0


,time_steps,first_block_unique,last_block_unique,lifetime_unique,total_txs_unique
address,,,,,
3CD1QW6fjgTwKq3Pj97nty28WZAVkziNom,47,1,1,1,1
1HtqDMWgn6186e8t3EesZQiw7gNbaPJfJH,43,1,1,1,1
1MEe2mebed8wopvy8xyjjHcEQHUPVJn2UC,39,1,1,1,1
1FAv42GaDuQixSzEzSbx6aP1Kf4WVWpQUY,39,1,1,1,1
18rdKmjrg1EawxgiVT3ikLExj6GWS2MNCk,38,1,1,1,1
1KumbRsTcA6UNqU2MHEfqEFnAZYU3w3izR,36,1,1,1,1
17BymcHaGRbXGnEzR2m9woUYNf9FBPPJ2P,35,1,1,1,1
1Po4J4SNyJuGnMGYJfGTXLEvGgAZKiddr7,35,1,1,1,1
1GUkazUBpXWdSJ9HbgTapAH7uybpi3Cs6K,35,1,1,1,1


In [97]:
# For each address, identify its earliest observed time step
# and compare its stored last appearance block with later records.

wallet_future_check = (
    wallets_features[
        ["address", "Time step",
         "first_block_appeared_in",
         "last_block_appeared_in",
         "lifetime_in_blocks",
         "total_txs"]
    ]
    .copy()
)

earliest_step = (
    wallet_future_check
    .groupby("address")["Time step"]
    .transform("min")
)

wallet_future_check["earliest_time_step"] = earliest_step

# Keep addresses that appear at more than one time step
wallet_future_check["multi_step_address"] = (
    wallet_future_check
    .groupby("address")["Time step"]
    .transform("nunique") > 1
)

multi_step_records = wallet_future_check[
    wallet_future_check["multi_step_address"]
].copy()

# For each address, obtain the last block stored at its earliest appearance
earliest_last_block = (
    multi_step_records[
        multi_step_records["Time step"] ==
        multi_step_records["earliest_time_step"]
    ]
    .groupby("address")["last_block_appeared_in"]
    .first()
)

multi_step_records["earliest_record_last_block"] = (
    multi_step_records["address"].map(earliest_last_block)
)

multi_step_records["same_last_block_as_earliest"] = np.isclose(
    multi_step_records["last_block_appeared_in"],
    multi_step_records["earliest_record_last_block"],
    equal_nan=True
)

print("Multi-time-step records:", len(multi_step_records))

print(
    "\nRecords retaining the same last_block as the earliest observation:"
)

print(
    multi_step_records["same_last_block_as_earliest"]
    .value_counts()
)

print(
    "\nPercentage retaining the same last_block:",
    multi_step_records["same_last_block_as_earliest"].mean() * 100
)

Multi-time-step records: 242974

Records retaining the same last_block as the earliest observation:
same_last_block_as_earliest
True    242974
Name: count, dtype: int64

Percentage retaining the same last_block: 100.0


In [98]:
# Address-history features in the inherited transaction feature set

address_history_features = [
    f"Local_feature_{i}" for i in range(23, 94)
]

print("Number of address-history features:",
      len(address_history_features))

print("\nAddress-history feature range:")
print(address_history_features[:5], "...", address_history_features[-5:])

print("\nTotal transaction features:",
      len(model_feature_cols))

Number of address-history features: 71

Address-history feature range:
['Local_feature_23', 'Local_feature_24', 'Local_feature_25', 'Local_feature_26', 'Local_feature_27'] ... ['Local_feature_89', 'Local_feature_90', 'Local_feature_91', 'Local_feature_92', 'Local_feature_93']

Total transaction features: 183


In [99]:
# Quantify Random Forest importance contributed by
# address-history features.

address_history_importance = (
    rf_importance[
        rf_importance["Feature"].isin(address_history_features)
    ]
    .copy()
)

address_history_importance = (
    address_history_importance
    .sort_values("Importance", ascending=False)
)

total_address_history_importance = (
    address_history_importance["Importance"].sum()
)

total_rf_importance = rf_importance["Importance"].sum()

print(
    "Address-history features:",
    len(address_history_importance)
)

print(
    "Total RF importance from address-history features:",
    total_address_history_importance
)

print(
    "Percentage of total RF importance:",
    total_address_history_importance /
    total_rf_importance * 100
)

print("\nTop 20 address-history features:")

display(
    address_history_importance.head(20)
)

Address-history features: 71
Total RF importance from address-history features: 0.5564295817801495
Percentage of total RF importance: 55.64295817801496

Top 20 address-history features:


,Feature,Importance,Is_Temporal_Graph_Derived
0,Local_feature_53,0.060273,True
1,Local_feature_41,0.047839,True
2,Local_feature_55,0.044676,True
4,Local_feature_43,0.039193,True
5,Local_feature_49,0.037444,True
6,Local_feature_47,0.033141,True
9,Local_feature_90,0.023141,True
13,Local_feature_65,0.016580,True
15,Local_feature_23,0.013726,True
17,Local_feature_52,0.013274,True


In [100]:
# Map address-history transaction features to their underlying
# wallet/address behavioural properties.

address_feature_mapping = {
    "Local_feature_23": "min received satoshi - input addresses",
    "Local_feature_24": "max received satoshi - input addresses",
    "Local_feature_25": "std received satoshi - input addresses",
    "Local_feature_26": "mean received satoshi - input addresses",
    "Local_feature_27": "Pearson correlation - received satoshi vs indices (input addresses)",
    "Local_feature_28": "Spearman correlation - received satoshi vs indices (input addresses)",
    "Local_feature_29": "min BTC sent - input addresses",
    "Local_feature_30": "max BTC sent - input addresses",
    "Local_feature_31": "std BTC sent - input addresses",
    "Local_feature_32": "mean BTC sent - input addresses",
    "Local_feature_33": "Pearson correlation - BTC sent vs indices (input addresses)",
    "Local_feature_34": "Spearman correlation - BTC sent vs indices (input addresses)",
    "Local_feature_35": "min available BTC - input addresses",
    "Local_feature_36": "max available BTC - input addresses",
    "Local_feature_37": "std available BTC - input addresses",
    "Local_feature_38": "mean available BTC - input addresses",
    "Local_feature_39": "Pearson correlation - available BTC vs indices (input addresses)",
    "Local_feature_40": "Spearman correlation - available BTC vs indices (input addresses)",
    "Local_feature_41": "min incoming transactions - input addresses",
    "Local_feature_42": "max incoming transactions - input addresses",
    "Local_feature_43": "std incoming transactions - input addresses",
    "Local_feature_44": "mean incoming transactions - input addresses",
    "Local_feature_45": "Pearson correlation - incoming transactions vs indices (input addresses)",
    "Local_feature_46": "Spearman correlation - incoming transactions vs indices (input addresses)",
    "Local_feature_47": "min outgoing transactions - input addresses",
    "Local_feature_48": "max outgoing transactions - input addresses",
    "Local_feature_49": "std outgoing transactions - input addresses",
    "Local_feature_50": "mean outgoing transactions - input addresses",
    "Local_feature_51": "Pearson correlation - outgoing transactions vs indices (input addresses)",
    "Local_feature_52": "Spearman correlation - outgoing transactions vs indices (input addresses)",
    "Local_feature_53": "min lifetime - input addresses",
    "Local_feature_54": "max lifetime - input addresses",
    "Local_feature_55": "std lifetime - input addresses",
    "Local_feature_56": "mean lifetime - input addresses",
    "Local_feature_57": "Pearson correlation - lifetime vs indices (input addresses)",
    "Local_feature_58": "Spearman correlation - lifetime vs indices (input addresses)",

    "Local_feature_59": "min received satoshi - output addresses",
    "Local_feature_60": "max received satoshi - output addresses",
    "Local_feature_61": "std received satoshi - output addresses",
    "Local_feature_62": "mean received satoshi - output addresses",
    "Local_feature_63": "Pearson correlation - received satoshi vs indices (output addresses)",
    "Local_feature_64": "Spearman correlation - received satoshi vs indices (output addresses)",
    "Local_feature_65": "min BTC sent - output addresses",
    "Local_feature_66": "max BTC sent - output addresses",
    "Local_feature_67": "std BTC sent - output addresses",
    "Local_feature_68": "mean BTC sent - output addresses",
    "Local_feature_69": "Pearson correlation - BTC sent vs indices (output addresses)",
    "Local_feature_70": "Spearman correlation - BTC sent vs indices (output addresses)",
    "Local_feature_71": "min available BTC - output addresses",
    "Local_feature_72": "max available BTC - output addresses",
    "Local_feature_73": "std available BTC - output addresses",
    "Local_feature_74": "mean available BTC - output addresses",
    "Local_feature_75": "Pearson correlation - available BTC vs indices (output addresses)",
    "Local_feature_76": "Spearman correlation - available BTC vs indices (output addresses)",
    "Local_feature_77": "min incoming transactions - output addresses",
    "Local_feature_78": "max incoming transactions - output addresses",
    "Local_feature_79": "std incoming transactions - output addresses",
    "Local_feature_80": "mean incoming transactions - output addresses",
    "Local_feature_81": "Pearson correlation - incoming transactions vs indices (output addresses)",
    "Local_feature_82": "Spearman correlation - incoming transactions vs indices (output addresses)",
    "Local_feature_83": "min outgoing transactions - output addresses",
    "Local_feature_84": "max outgoing transactions - output addresses",
    "Local_feature_85": "std outgoing transactions - output addresses",
    "Local_feature_86": "mean outgoing transactions - output addresses",
    "Local_feature_87": "Pearson correlation - outgoing transactions vs indices (output addresses)",
    "Local_feature_88": "Spearman correlation - outgoing transactions vs indices (output addresses)",
    "Local_feature_89": "min lifetime - output addresses",
    "Local_feature_90": "max lifetime - output addresses",
    "Local_feature_91": "std lifetime - output addresses",
    "Local_feature_92": "mean lifetime - output addresses",
    "Local_feature_93": "Pearson/Spearman lifetime statistic - output addresses"
}

print("Mapped features:", len(address_feature_mapping))

display(
    pd.DataFrame(
        list(address_feature_mapping.items()),
        columns=["Feature", "Wallet_Property"]
    )
)

Mapped features: 71


,Feature,Wallet_Property
0,Local_feature_23,min received satoshi - input addresses
1,Local_feature_24,max received satoshi - input addresses
2,Local_feature_25,std received satoshi - input addresses
3,Local_feature_26,mean received satoshi - input addresses
4,Local_feature_27,Pearson correlation - received satoshi vs indi...
5,Local_feature_28,Spearman correlation - received satoshi vs ind...
6,Local_feature_29,min BTC sent - input addresses
7,Local_feature_30,max BTC sent - input addresses
8,Local_feature_31,std BTC sent - input addresses
9,Local_feature_32,mean BTC sent - input addresses


In [101]:
# Combine exact feature mapping with Random Forest importance.

address_audit = address_history_importance.copy()

address_audit["Wallet_Property"] = (
    address_audit["Feature"].map(address_feature_mapping)
)

address_audit = address_audit[
    ["Feature", "Wallet_Property", "Importance"]
].sort_values(
    "Importance",
    ascending=False
)

print("Address-history feature audit:")
display(address_audit)

print("\nFeatures without a mapped property:")
display(
    address_audit[address_audit["Wallet_Property"].isna()]
)

Address-history feature audit:


,Feature,Wallet_Property,Importance
0,Local_feature_53,min lifetime - input addresses,0.060273
1,Local_feature_41,min incoming transactions - input addresses,0.047839
2,Local_feature_55,std lifetime - input addresses,0.044676
4,Local_feature_43,std incoming transactions - input addresses,0.039193
5,Local_feature_49,std outgoing transactions - input addresses,0.037444
6,Local_feature_47,min outgoing transactions - input addresses,0.033141
9,Local_feature_90,max lifetime - output addresses,0.023141
13,Local_feature_65,min BTC sent - output addresses,0.016580
15,Local_feature_23,min received satoshi - input addresses,0.013726
17,Local_feature_52,Spearman correlation - outgoing transactions v...,0.013274



Features without a mapped property:


,Feature,Wallet_Property,Importance


In [102]:
# Group address-history features by the underlying wallet behaviour.

feature_groups = {}

for feature, prop in address_feature_mapping.items():
    if prop is None:
        feature_groups[feature] = "Unverified"
    elif "lifetime" in prop:
        feature_groups[feature] = "Lifetime"
    elif "incoming transactions" in prop:
        feature_groups[feature] = "Transaction_Count_In"
    elif "outgoing transactions" in prop:
        feature_groups[feature] = "Transaction_Count_Out"
    elif "available BTC" in prop:
        feature_groups[feature] = "Available_BTC"
    elif "BTC sent" in prop:
        feature_groups[feature] = "BTC_Sent"
    elif "received satoshi" in prop:
        feature_groups[feature] = "BTC_Received"
    else:
        feature_groups[feature] = "Other"

address_audit["Feature_Group"] = (
    address_audit["Feature"].map(feature_groups)
)

group_summary = (
    address_audit
    .groupby("Feature_Group")
    .agg(
        Feature_Count=("Feature", "count"),
        Total_Importance=("Importance", "sum"),
        Mean_Importance=("Importance", "mean")
    )
    .sort_values("Total_Importance", ascending=False)
)

display(group_summary)

,Feature_Count,Total_Importance,Mean_Importance
Feature_Group,,,
Lifetime,11,0.152645,0.013877
Transaction_Count_In,12,0.135873,0.011323
Transaction_Count_Out,12,0.111890,0.009324
BTC_Received,12,0.072395,0.006033
BTC_Sent,12,0.055475,0.004623
Available_BTC,12,0.028152,0.002346


In [103]:
# Detailed view of the address-history features,
# grouped by their underlying wallet behaviour.

display(
    address_audit[
        ["Feature", "Feature_Group", "Wallet_Property", "Importance"]
    ]
    .sort_values(
        ["Feature_Group", "Importance"],
        ascending=[True, False]
    )
)

,Feature,Feature_Group,Wallet_Property,Importance
20,Local_feature_76,Available_BTC,Spearman correlation - available BTC vs indice...,0.011908
24,Local_feature_40,Available_BTC,Spearman correlation - available BTC vs indice...,0.010676
114,Local_feature_73,Available_BTC,std available BTC - output addresses,0.001157
119,Local_feature_71,Available_BTC,min available BTC - output addresses,0.000999
122,Local_feature_72,Available_BTC,max available BTC - output addresses,0.000929
134,Local_feature_75,Available_BTC,Pearson correlation - available BTC vs indices...,0.000721
138,Local_feature_74,Available_BTC,mean available BTC - output addresses,0.000680
150,Local_feature_37,Available_BTC,std available BTC - input addresses,0.000494
162,Local_feature_35,Available_BTC,min available BTC - input addresses,0.000320
177,Local_feature_36,Available_BTC,max available BTC - input addresses,0.000170


In [104]:
# Check whether address-history-derived transaction features
# remain identical across repeated observations of the same address.

# We need the transaction -> address relationships first.
addr_tx = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\AddrTx_edgelist.csv")
tx_addr = pd.read_csv(r"C:\Users\koush\OneDrive\Documents\Financial_Forensics\data\raw\eliptic++\TxAddr_edgelist.csv")

print("AddrTx shape:", addr_tx.shape)
print("TxAddr shape:", tx_addr.shape)

print("\nColumns:")
print("AddrTx:", addr_tx.columns.tolist())
print("TxAddr:", tx_addr.columns.tolist())

AddrTx shape: (477117, 2)
TxAddr shape: (837124, 2)

Columns:
AddrTx: ['input_address', 'txId']
TxAddr: ['txId', 'output_address']


In [105]:
# Display a few rows to understand how transaction and address IDs
# are represented in the two edge files.

print("AddrTx sample:")
display(addr_tx.head())

print("\nTxAddr sample:")
display(tx_addr.head())

print("\nTransaction feature sample:")
display(
    tx_features[
        ["txId", "Time step"] +
        address_history_features[:10]
    ].head()
)

AddrTx sample:


,input_address,txId
0,14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a,230325127
1,13Lhad3SAmu2vqYg2dxbNcxH7LE77kJu2w,230325139
2,1MAQQZn7EHP6J3erXByCciFiVcgS8ZhWqz,86875675
3,16zs5SVSyADh5WrLNbZbpRLsBsN5uEzgeK,230325147
4,1QJpwtUorBKPGUJkSyrRcBKTAHq4CXrdYh,230325154



TxAddr sample:


,txId,output_address
0,230325127,1GASxu5nMntiRKdVtTVRvEbP965G51bhHH
1,230325127,14YRXHHof4BY1TVxN5FqYPcEdpmXiYT78a
2,230325139,1GFdrdgtG34GChM8SMpMwcXFc4nYbH1A5G
3,86875675,19q57SeCEzTnWrWVXA43nZzhSiXkYggh7c
4,86875675,1Kk1NVYnCE8ALXDhgMM6HqTt1jDSvi6QBA



Transaction feature sample:


,txId,Time step,Local_feature_23,Local_feature_24,Local_feature_25,Local_feature_26,Local_feature_27,Local_feature_28,Local_feature_29,Local_feature_30,Local_feature_31,Local_feature_32
0,3321,1,-0.148869,-0.080147,-0.155604,-0.010763,-0.012107,-0.139665,-0.148864,-0.080147,-0.155604,-0.010669
1,11108,1,-0.148805,-0.080147,-0.155520,-0.010763,-0.012107,-0.139566,-0.148800,-0.080147,-0.155520,-0.010669
2,51816,1,-0.148912,-0.080147,-0.155661,-0.010763,-0.012107,-0.139732,-0.148907,-0.080147,-0.155661,-0.010669
3,68869,1,-0.148638,-0.080147,-0.155297,-0.010763,-0.012107,-0.139303,-0.148633,-0.080147,-0.155297,-0.010669
4,89273,1,1.107041,-0.080147,1.512162,-0.010763,-0.012107,1.828864,1.107130,-0.080147,1.512340,-0.010669


In [106]:
# Build address lists for every transaction

input_addresses_by_tx = (
    addr_tx.groupby("txId")["input_address"]
    .agg(list)
    .rename("input_addresses")
)

output_addresses_by_tx = (
    tx_addr.groupby("txId")["output_address"]
    .agg(list)
    .rename("output_addresses")
)

tx_address_map = pd.concat(
    [input_addresses_by_tx, output_addresses_by_tx],
    axis=1
).reset_index()

print("Transactions with input addresses :", input_addresses_by_tx.shape[0])
print("Transactions with output addresses:", output_addresses_by_tx.shape[0])

display(tx_address_map.head())

Transactions with input addresses : 202804
Transactions with output addresses: 202804


,txId,input_addresses,output_addresses
0,1076,"[18LEv2TV1sg1GkGd6w5FdD3vp1dmiTTcNr, 1LXcdzQpi...","[1DfE5KELWYYhgDsNcvhTQwMUHLtsz32Kj, 3D28Kvhgbb..."
1,2534,"[149hEPHkEWH31qaMRz77tFxjk3WrQkT1P1, 1AYHBouG6...","[1PDf2CkNLa7TkZmQSW3njkkoMHZpU1GVXF, 1MFt5JfB6..."
2,3181,[1GX28yLjVWux7ws4UQ9FB4MnLH4UKTPK2z],"[1GKkMWXLHXVVfv9RY1oZDtuFEmo4Xf3m3V, 1D9DWDH1b..."
3,3321,[18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB],"[18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB, 1EhgvhK2N..."
4,3889,[1LjYX8JCSSCgzUdEHVZi4qcdxzn7YEkr1M],"[1DovMcRhEoXpsaG4xod9RnuBZ3K6rtLosR, 19rK6LmWG..."


In [107]:
# Attach the transaction time step so that we can later compare
# wallet information at the corresponding transaction time.

tx_address_map = tx_address_map.merge(
    tx_features[["txId", "Time step"]],
    on="txId",
    how="left"
)

print("Shape:", tx_address_map.shape)

print("\nMissing Time step:")
print(tx_address_map["Time step"].isna().sum())

display(tx_address_map.head())

Shape: (202804, 4)

Missing Time step:
0


,txId,input_addresses,output_addresses,Time step
0,1076,"[18LEv2TV1sg1GkGd6w5FdD3vp1dmiTTcNr, 1LXcdzQpi...","[1DfE5KELWYYhgDsNcvhTQwMUHLtsz32Kj, 3D28Kvhgbb...",48
1,2534,"[149hEPHkEWH31qaMRz77tFxjk3WrQkT1P1, 1AYHBouG6...","[1PDf2CkNLa7TkZmQSW3njkkoMHZpU1GVXF, 1MFt5JfB6...",6
2,3181,[1GX28yLjVWux7ws4UQ9FB4MnLH4UKTPK2z],"[1GKkMWXLHXVVfv9RY1oZDtuFEmo4Xf3m3V, 1D9DWDH1b...",34
3,3321,[18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB],"[18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB, 1EhgvhK2N...",1
4,3889,[1LjYX8JCSSCgzUdEHVZi4qcdxzn7YEkr1M],"[1DovMcRhEoXpsaG4xod9RnuBZ3K6rtLosR, 19rK6LmWG...",48


In [108]:
# Keep only the wallet fields needed for the first leakage test.
# Local_feature_53 = minimum lifetime of INPUT addresses.

wallet_lifetime = wallets_features[
    ["address", "Time step", "lifetime_in_blocks"]
].copy()

print("Wallet lifetime records:", wallet_lifetime.shape)

print("\nUnique addresses:",
      wallet_lifetime["address"].nunique())

print("\nTime-step range:",
      wallet_lifetime["Time step"].min(),
      "to",
      wallet_lifetime["Time step"].max())

display(wallet_lifetime.head())

Wallet lifetime records: (1268260, 3)

Unique addresses: 822942

Time-step range: 1 to 49


,address,Time step,lifetime_in_blocks
0,111112TykSw72ztDN2WJger4cynzWYC5w,25,0.0
1,1111DAYXhoxZx2tsRnzimfozo783x1yC2,25,46370.0
2,1111DAYXhoxZx2tsRnzimfozo783x1yC2,29,46370.0
3,1111DAYXhoxZx2tsRnzimfozo783x1yC2,39,46370.0
4,1111DAYXhoxZx2tsRnzimfozo783x1yC2,39,46370.0


In [109]:
# Expand each transaction so that every input address becomes one row.
# Then attach the wallet lifetime recorded for that address at the
# transaction's corresponding time step.

input_long = tx_address_map[
    ["txId", "Time step", "input_addresses"]
].explode("input_addresses")

input_long = input_long.rename(
    columns={"input_addresses": "address"}
)

input_long = input_long.merge(
    wallet_lifetime,
    on=["address", "Time step"],
    how="left"
)

print("Expanded input-address rows:", input_long.shape)

print("\nMissing wallet lifetime values:",
      input_long["lifetime_in_blocks"].isna().sum())

display(input_long.head(10))

Expanded input-address rows: (4686055, 4)

Missing wallet lifetime values: 0


,txId,Time step,address,lifetime_in_blocks
0,1076,48,18LEv2TV1sg1GkGd6w5FdD3vp1dmiTTcNr,0.0
1,1076,48,1LXcdzQpiWoaAVvRRETdWmS4YXAe7qD3Nq,0.0
2,1076,48,1LXcdzQpiWoaAVvRRETdWmS4YXAe7qD3Nq,0.0
3,2534,6,149hEPHkEWH31qaMRz77tFxjk3WrQkT1P1,0.0
4,2534,6,1AYHBouG6FYjyyJJoj6s4gP8RxKFJZBKk8,0.0
5,2534,6,18QvbfmFUcYuUuptrhksyti5eMs7ANeQgG,0.0
6,3181,34,1GX28yLjVWux7ws4UQ9FB4MnLH4UKTPK2z,94754.0
7,3181,34,1GX28yLjVWux7ws4UQ9FB4MnLH4UKTPK2z,94754.0
8,3321,1,18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB,6055.0
9,3321,1,18TnVVsNqViRGTNzCjk5dnW4VNxpehywXB,6055.0


In [110]:
# Reconstruct the minimum lifetime of the input addresses for each transaction.

reconstructed_lifetime = (
    input_long
    .groupby("txId")["lifetime_in_blocks"]
    .min()
    .rename("reconstructed_min_input_lifetime")
    .reset_index()
)

print("Transactions reconstructed:",
      reconstructed_lifetime.shape[0])

display(reconstructed_lifetime.head())

Transactions reconstructed: 202804


,txId,reconstructed_min_input_lifetime
0,1076,0.0
1,2534,0.0
2,3181,94754.0
3,3321,6055.0
4,3889,0.0


In [111]:
from scipy.stats import spearmanr, pearsonr

# Get the transaction feature corresponding to:
# Local_feature_53 = minimum lifetime of input addresses

feature_53 = tx_features[
    ["txId", "Local_feature_53"]
].copy()

lifetime_comparison = reconstructed_lifetime.merge(
    feature_53,
    on="txId",
    how="inner"
)

print("Comparison rows:", lifetime_comparison.shape[0])

# Correlation between reconstructed raw lifetime and stored feature
pearson_corr, pearson_p = pearsonr(
    lifetime_comparison["reconstructed_min_input_lifetime"],
    lifetime_comparison["Local_feature_53"]
)

spearman_corr, spearman_p = spearmanr(
    lifetime_comparison["reconstructed_min_input_lifetime"],
    lifetime_comparison["Local_feature_53"]
)

print("\nCorrelation with Local_feature_53")
print("--------------------------------")
print(f"Pearson correlation  : {pearson_corr:.6f}")
print(f"Spearman correlation : {spearman_corr:.6f}")

display(lifetime_comparison.head(20))

Comparison rows: 202804

Correlation with Local_feature_53
--------------------------------
Pearson correlation  : 0.637222
Spearman correlation : 0.450513


,txId,reconstructed_min_input_lifetime,Local_feature_53
0,1076,0.0,-0.488060
1,2534,0.0,0.547344
2,3181,94754.0,4.902787
3,3321,6055.0,1.378639
4,3889,0.0,0.063613
5,5473,0.0,-0.488362
6,6415,4.0,-0.488339
7,6418,2.0,2.487477
8,7952,94754.0,4.902787
9,7954,0.0,-0.487457


In [112]:
# For addresses appearing at multiple time steps, compare the
# lifetime recorded at their earliest observation with their latest observation.

wallet_lifetime_range = (
    wallets_features
    .groupby("address")["lifetime_in_blocks"]
    .agg(
        observations="count",
        min_lifetime="min",
        max_lifetime="max",
        first_lifetime="first",
        last_lifetime="last"
    )
    .reset_index()
)

multi_wallets = wallet_lifetime_range[
    wallet_lifetime_range["observations"] > 1
].copy()

print("Addresses appearing at multiple time steps:",
      len(multi_wallets))

print(
    "Addresses with unchanged lifetime:",
    (
        multi_wallets["min_lifetime"]
        == multi_wallets["max_lifetime"]
    ).sum()
)

print(
    "Percentage unchanged:",
    round(
        (
            multi_wallets["min_lifetime"]
            == multi_wallets["max_lifetime"]
        ).mean() * 100,
        2
    ),
    "%"
)

display(multi_wallets.head(20))

Addresses appearing at multiple time steps: 273268
Addresses with unchanged lifetime: 273268
Percentage unchanged: 100.0 %


,address,observations,min_lifetime,max_lifetime,first_lifetime,last_lifetime
1,1111DAYXhoxZx2tsRnzimfozo783x1yC2,8,46370.0,46370.0,46370.0,46370.0
3,111218KKkh1JJFRHbwM16AwCiVCc4m7he1,2,0.0,0.0,0.0,0.0
5,1117wASFaYgJJP6MiY8cPD5DMdQda8gDZ,2,2.0,2.0,2.0,2.0
10,111Ud9zxFi3VgcstEM2R3YAS3CNxf9o8r,2,1.0,1.0,1.0,1.0
11,111Y3BkUGLSWQjHCb2Mg8oMNiwc1jxMwe,3,4.0,4.0,4.0,4.0
12,111YiWPqpWjz1Y8RNy3JGKq1CdQpp841n,2,1.0,1.0,1.0,1.0
14,111iu559diy1cBDTP9cRhy8sYYcECvvrw,2,1.0,1.0,1.0,1.0
15,111khWGs3Mj7UgKT7aS6hLsYK11ii4Gmw,5,40324.0,40324.0,40324.0,40324.0
17,11214iNCgVdqCtqeLiYUNHtwiBeVx9U56X,2,1.0,1.0,1.0,1.0
20,1121A3vrYYduVPMnfS87ieHFP2bwy6xriv,4,66523.0,66523.0,66523.0,66523.0


In [113]:
# Check whether last_block_appeared_in also remains fixed across
# repeated observations of the same address.

wallet_history_check = (
    wallets_features
    .groupby("address")
    .agg(
        observations=("Time step", "count"),
        first_time=("Time step", "min"),
        last_time=("Time step", "max"),
        first_block_min=("first_block_appeared_in", "min"),
        first_block_max=("first_block_appeared_in", "max"),
        last_block_min=("last_block_appeared_in", "min"),
        last_block_max=("last_block_appeared_in", "max"),
        lifetime_min=("lifetime_in_blocks", "min"),
        lifetime_max=("lifetime_in_blocks", "max"),
        total_txs_min=("total_txs", "min"),
        total_txs_max=("total_txs", "max")
    )
    .reset_index()
)

multi_history = wallet_history_check[
    wallet_history_check["observations"] > 1
].copy()

print("Repeated addresses:", len(multi_history))

print("\nUnchanged last_block:",
      (
          multi_history["last_block_min"]
          == multi_history["last_block_max"]
      ).sum())

print("Unchanged lifetime:",
      (
          multi_history["lifetime_min"]
          == multi_history["lifetime_max"]
      ).sum())

print("Unchanged total_txs:",
      (
          multi_history["total_txs_min"]
          == multi_history["total_txs_max"]
      ).sum())

display(multi_history.head(20))

Repeated addresses: 273268

Unchanged last_block: 273268
Unchanged lifetime: 273268
Unchanged total_txs: 273268


,address,observations,first_time,last_time,first_block_min,first_block_max,last_block_min,last_block_max,lifetime_min,lifetime_max,total_txs_min,total_txs_max
1,1111DAYXhoxZx2tsRnzimfozo783x1yC2,8,25,48,439589.0,439589.0,485959.0,485959.0,46370.0,46370.0,8.0,8.0
3,111218KKkh1JJFRHbwM16AwCiVCc4m7he1,2,17,17,423456.0,423456.0,423456.0,423456.0,0.0,0.0,2.0,2.0
5,1117wASFaYgJJP6MiY8cPD5DMdQda8gDZ,2,5,5,399269.0,399269.0,399271.0,399271.0,2.0,2.0,2.0,2.0
10,111Ud9zxFi3VgcstEM2R3YAS3CNxf9o8r,2,33,33,455714.0,455714.0,455715.0,455715.0,1.0,1.0,2.0,2.0
11,111Y3BkUGLSWQjHCb2Mg8oMNiwc1jxMwe,3,36,36,461765.0,461765.0,461769.0,461769.0,4.0,4.0,3.0,3.0
12,111YiWPqpWjz1Y8RNy3JGKq1CdQpp841n,2,4,4,397248.0,397248.0,397249.0,397249.0,1.0,1.0,2.0,2.0
14,111iu559diy1cBDTP9cRhy8sYYcECvvrw,2,32,32,453699.0,453699.0,453700.0,453700.0,1.0,1.0,2.0,2.0
15,111khWGs3Mj7UgKT7aS6hLsYK11ii4Gmw,5,14,34,417408.0,417408.0,457732.0,457732.0,40324.0,40324.0,5.0,5.0
17,11214iNCgVdqCtqeLiYUNHtwiBeVx9U56X,2,13,13,415392.0,415392.0,415393.0,415393.0,1.0,1.0,2.0,2.0
20,1121A3vrYYduVPMnfS87ieHFP2bwy6xriv,4,9,42,407342.0,407342.0,473865.0,473865.0,66523.0,66523.0,4.0,4.0


In [114]:
# Identify addresses where the dataset's recorded history extends
# substantially beyond the first time step at which the address appears.

address_temporal_span = (
    wallets_features
    .groupby("address")
    .agg(
        first_time_step=("Time step", "min"),
        last_time_step=("Time step", "max"),
        first_block=("first_block_appeared_in", "first"),
        last_block=("last_block_appeared_in", "first"),
        lifetime=("lifetime_in_blocks", "first"),
        total_txs=("total_txs", "first")
    )
    .reset_index()
)

address_temporal_span["time_span"] = (
    address_temporal_span["last_time_step"]
    - address_temporal_span["first_time_step"]
)

long_history_addresses = (
    address_temporal_span[
        address_temporal_span["time_span"] >= 10
    ]
    .sort_values("time_span", ascending=False)
)

print(
    "Addresses appearing across >=10 time steps:",
    len(long_history_addresses)
)

display(long_history_addresses.head(20))

Addresses appearing across >=10 time steps: 24114


,address,first_time_step,last_time_step,first_block,last_block,lifetime,total_txs,time_span
812765,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,1,49,391202.0,487969.0,96767.0,37.0,48
821601,3QkXtcSWJA9w77eCujnMBKDWFe7F7zwxTg,1,49,391205.0,487975.0,96770.0,40.0,48
815912,3Pei4hHgGpXQAJFremyy3yNREcLzvBvh4E,1,49,391200.0,487969.0,96769.0,36.0,48
819801,3QQKRwuVyRgv7xZVN9aR1z8p7SUFukAsCz,1,49,391211.0,487969.0,96758.0,22.0,48
822620,3Qy5mTpumWh24f5UdwGPJEhStoD3A25fK9,1,49,391211.0,487969.0,96758.0,40.0,48
648366,1P6sK32XSime89NY88ZL3PtvJGDfrytYPc,1,49,391202.0,487971.0,96769.0,13.0,48
805802,3MfN5to5K5be2RupWE8rjJHQ6V9L8ypWeh,1,49,391202.0,487975.0,96773.0,208.0,48
806274,3MmajzyU2LFun5Xc4N6rmVYH7tnGxc8bqM,1,49,391202.0,487969.0,96767.0,30.0,48
711374,33AibmPKKKDRfSZsqgugzGY5MQnrVYpfQr,1,49,391200.0,487969.0,96769.0,35.0,48
708044,32XUXaJftyDXMqkQt81GJmwCP513SohC1W,1,49,391206.0,487969.0,96763.0,26.0,48


In [115]:
# Pick an address whose first observation is much earlier than its
# final observation and display every recorded wallet row.

if len(long_history_addresses) > 0:

    example_address = long_history_addresses.iloc[0]["address"]

    example_history = (
        wallets_features[
            wallets_features["address"] == example_address
        ][
            [
                "address",
                "Time step",
                "first_block_appeared_in",
                "last_block_appeared_in",
                "lifetime_in_blocks",
                "total_txs"
            ]
        ]
        .sort_values("Time step")
    )

    print("Example address:", example_address)
    display(example_history)

Example address: 3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G


,address,Time step,first_block_appeared_in,last_block_appeared_in,lifetime_in_blocks,total_txs
1251862,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,1,391202.0,487969.0,96767.0,37.0
1251863,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,1,391202.0,487969.0,96767.0,37.0
1251864,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,2,391202.0,487969.0,96767.0,37.0
1251865,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,2,391202.0,487969.0,96767.0,37.0
1251866,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,5,391202.0,487969.0,96767.0,37.0
1251867,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,5,391202.0,487969.0,96767.0,37.0
1251868,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,5,391202.0,487969.0,96767.0,37.0
1251869,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,5,391202.0,487969.0,96767.0,37.0
1251870,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,7,391202.0,487969.0,96767.0,37.0
1251871,3P2eoKr3vAeZhJcTzon3VFkv5r7DqSXW9G,7,391202.0,487969.0,96767.0,37.0


In [116]:
# Reconstruct several input-address wallet statistics at transaction level.
# These correspond to important Local_features identified in the RF audit.

wallet_stats = wallets_features[
    [
        "address",
        "Time step",
        "lifetime_in_blocks",
        "num_txs_as_sender",
        "num_txs_as receiver",
        "btc_received_total",
        "btc_sent_total"
    ]
].copy()

# Attach wallet information to every input-address observation
input_wallet = input_long[
    ["txId", "Time step", "address"]
].merge(
    wallet_stats,
    on=["address", "Time step"],
    how="left"
)

# Aggregate wallet properties across input addresses of each transaction
reconstructed_wallet_features = (
    input_wallet
    .groupby("txId")
    .agg(
        min_lifetime=("lifetime_in_blocks", "min"),
        max_lifetime=("lifetime_in_blocks", "max"),
        std_lifetime=("lifetime_in_blocks", "std"),

        min_txs_in=("num_txs_as receiver", "min"),
        max_txs_in=("num_txs_as receiver", "max"),
        std_txs_in=("num_txs_as receiver", "std"),

        min_txs_out=("num_txs_as_sender", "min"),
        max_txs_out=("num_txs_as_sender", "max"),
        std_txs_out=("num_txs_as_sender", "std"),

        min_btc_received=("btc_received_total", "min"),
        max_btc_received=("btc_received_total", "max"),
        mean_btc_received=("btc_received_total", "mean"),

        min_btc_sent=("btc_sent_total", "min"),
        max_btc_sent=("btc_sent_total", "max"),
        mean_btc_sent=("btc_sent_total", "mean")
    )
    .reset_index()
)

print("Reconstructed transaction rows:",
      reconstructed_wallet_features.shape[0])

display(reconstructed_wallet_features.head())

MemoryError: Unable to allocate 6.63 GiB for an array with shape (889752851,) and data type int64

### Kernel state note (2026-09-18)

Cell 108 crashed with a `MemoryError` merging `input_long` with `wallet_stats`. Diagnosed outside the notebook (clean process, identical cell 96-108 logic) and found two separate issues:

1. **`wallets_features.csv` has duplicate `(address, Time step)` rows** -- up to 568 duplicate rows for one address in one time step. All 5 wallet fields needed here are 100% identical within every duplicate group, so `drop_duplicates(subset=["address","Time step"])` before merging is safe and lossless.
2. **The live kernel's `input_long` was ~9.8x inflated** (4,686,055 rows here vs. a clean re-run's 477,117, which matches `AddrTx_edgelist.csv` exactly and the earlier-computed mean of 2.3526 input addresses/tx x 202,804 tx). This was stale/out-of-order kernel state, not a data issue -- fixed by restarting the kernel and re-running top to bottom.

The cells below (and the corrected comparison cell that follows) contain code and outputs verified against a clean run of the real CSVs. **Run these after a kernel restart** so your live variables match.

In [ ]:
# The previous cell crashed with a MemoryError while merging input_long
# with wallet_stats on [address, Time step]. This indicates a many-to-many
# join: wallets_features can contain multiple rows for the same address
# within the same time step (one row per transaction it touched that step).
# Confirm this, and check whether the repeated rows carry identical values
# (safe to collapse) or differ (would need a different aggregation).

dup_group_sizes = wallet_stats.groupby(["address", "Time step"]).size()

print("Total (address, Time step) groups:", dup_group_sizes.shape[0])
print("Groups with more than 1 row     :", (dup_group_sizes > 1).sum())
print("Largest duplicate group size    :", dup_group_sizes.max())

value_cols = [
    "lifetime_in_blocks",
    "num_txs_as_sender",
    "num_txs_as receiver",
    "btc_received_total",
    "btc_sent_total"
]

nunique_per_group = wallet_stats.groupby(["address", "Time step"])[value_cols].nunique()
dup_mask = (dup_group_sizes > 1).values
all_identical = (nunique_per_group[dup_mask] == 1).all(axis=1)

print("\nFraction of duplicate groups where all wallet fields are identical:")
print(round(all_identical.mean(), 4), f"({all_identical.sum()} / {len(all_identical)})")

Total (address, Time step) groups: 920691
Groups with more than 1 row     : 246545
Largest duplicate group size    : 568

Fraction of duplicate groups where all wallet fields are identical:
1.0 (246545 / 246545)


In [ ]:
# Deduplicate to one row per (address, Time step) before merging.
# The previous cell confirmed the duplicate rows are identical on every
# field we need, so this only removes the one-row-per-transaction
# repetition that caused the merge to explode -- no information is lost.

wallet_stats_dedup = wallet_stats.drop_duplicates(subset=["address", "Time step"])

print("wallet_stats rows before dedup:", wallet_stats.shape[0])
print("wallet_stats rows after dedup :", wallet_stats_dedup.shape[0])

input_wallet = input_long[
    ["txId", "Time step", "address"]
].merge(
    wallet_stats_dedup,
    on=["address", "Time step"],
    how="left"
)

print("\ninput_wallet shape:", input_wallet.shape)
print("Expected row count (should match input_long):", input_long.shape[0])

reconstructed_wallet_features = (
    input_wallet
    .groupby("txId")
    .agg(
        min_lifetime=("lifetime_in_blocks", "min"),
        max_lifetime=("lifetime_in_blocks", "max"),
        std_lifetime=("lifetime_in_blocks", "std"),

        min_txs_in=("num_txs_as receiver", "min"),
        max_txs_in=("num_txs_as receiver", "max"),
        std_txs_in=("num_txs_as receiver", "std"),

        min_txs_out=("num_txs_as_sender", "min"),
        max_txs_out=("num_txs_as_sender", "max"),
        std_txs_out=("num_txs_as_sender", "std"),

        min_btc_received=("btc_received_total", "min"),
        max_btc_received=("btc_received_total", "max"),
        mean_btc_received=("btc_received_total", "mean"),

        min_btc_sent=("btc_sent_total", "min"),
        max_btc_sent=("btc_sent_total", "max"),
        mean_btc_sent=("btc_sent_total", "mean")
    )
    .reset_index()
)

print("\nReconstructed transaction rows:", reconstructed_wallet_features.shape[0])
display(reconstructed_wallet_features.head())

wallet_stats rows before dedup: 1268260
wallet_stats rows after dedup : 920691

input_wallet shape: (477117, 8)
Expected row count (should match input_long): 477117

Reconstructed transaction rows: 202804


   txId  min_lifetime  max_lifetime  std_lifetime  min_txs_in  max_txs_in  std_txs_in  min_txs_out  max_txs_out  std_txs_out  min_btc_received  max_btc_received  mean_btc_received  min_btc_sent  max_btc_sent  mean_btc_sent
0  1076           0.0           0.0           0.0         0.0         1.0    0.707107          1.0          1.0          0.0          0.000000          0.704040           0.352020      0.008543      0.704040       0.356291
1  2534           0.0           0.0           0.0         0.0         0.0    0.000000          1.0          1.0          0.0          0.000000          0.000000           0.000000      0.011462      0.267733       0.113651
2  3181       94754.0       94754.0           NaN         6.0         6.0         NaN          6.0          6.0          NaN       2740.300000       2740.300000        2740.300000   2740.300000   2740.300000    2740.300000
3  3321        6055.0        6055.0           NaN         2.0         2.0         NaN          2.0          

In [ ]:
# Compare reconstructed wallet-history statistics against the
# corresponding transaction-level Local_features.

comparison = reconstructed_wallet_features.merge(
    tx_features[[
        "txId",
        "Local_feature_53",  # min lifetime - input addresses
        "Local_feature_54",  # max lifetime - input addresses
        "Local_feature_55",  # std lifetime - input addresses
        "Local_feature_41",  # min incoming transactions - input addresses
        "Local_feature_42",  # max incoming transactions - input addresses
        "Local_feature_43",  # std incoming transactions - input addresses
        "Local_feature_47",  # min outgoing transactions - input addresses
        "Local_feature_48",  # max outgoing transactions - input addresses
        "Local_feature_49",  # std outgoing transactions - input addresses
        "Local_feature_23",  # min received satoshi - input addresses
        "Local_feature_24",  # max received satoshi - input addresses
        "Local_feature_29",  # min BTC sent - input addresses
        "Local_feature_30",  # max BTC sent - input addresses
    ]],
    on="txId",
    how="inner"
)

feature_pairs = [
    ("min_lifetime", "Local_feature_53"),
    ("max_lifetime", "Local_feature_54"),
    ("std_lifetime", "Local_feature_55"),
    ("min_txs_in", "Local_feature_41"),
    ("max_txs_in", "Local_feature_42"),
    ("std_txs_in", "Local_feature_43"),
    ("min_txs_out", "Local_feature_47"),
    ("max_txs_out", "Local_feature_48"),
    ("std_txs_out", "Local_feature_49"),
    ("min_btc_received", "Local_feature_23"),
    ("max_btc_received", "Local_feature_24"),
    ("min_btc_sent", "Local_feature_29"),
    ("max_btc_sent", "Local_feature_30")
]

results = []

for raw_feature, stored_feature in feature_pairs:
    valid = comparison[[raw_feature, stored_feature]].dropna()

    if len(valid) > 1:
        pearson = valid[raw_feature].corr(valid[stored_feature])
        spearman = valid[raw_feature].corr(
            valid[stored_feature],
            method="spearman"
        )
    else:
        pearson = np.nan
        spearman = np.nan

    results.append({
        "Raw_Wallet_Statistic": raw_feature,
        "Transaction_Feature": stored_feature,
        "Pearson": pearson,
        "Spearman": spearman,
        "Valid_Rows": len(valid)
    })

correlation_results = pd.DataFrame(results)

display(
    correlation_results.sort_values(
        "Spearman",
        ascending=False
    )
)

Comparison rows: 202804


   Raw_Wallet_Statistic Transaction_Feature   Pearson  Spearman  Valid_Rows
8           std_txs_out    Local_feature_49  0.527531  0.764375       43507
6           min_txs_out    Local_feature_47  0.819177  0.753864      202804
11         min_btc_sent    Local_feature_29  0.631611  0.677620      202804
5            std_txs_in    Local_feature_43  0.330126  0.601340       43507
2          std_lifetime    Local_feature_55  0.538302  0.593816       43507
0          min_lifetime    Local_feature_53  0.637222  0.450513      202804
9      min_btc_received    Local_feature_23  0.608798  0.349362      202804
4            max_txs_in    Local_feature_42  0.351179  0.240214      202804
7           max_txs_out    Local_feature_48  0.154244  0.187086      202804
1          max_lifetime    Local_feature_54  0.434592  0.179294      202804
3            min_txs_in    Local_feature_41  0.254816  0.087239      202804
10     max_btc_received    Local_feature_24  0.629238  0.010917      202804
12         m

### Feature construction audit: does the documented mapping hold?

The correlations above (0.01-0.82, no clean 1:1 matches) rule out "`Local_feature_X` is a rescaled copy of the obvious raw wallet statistic." Instead of guessing more mappings from the recovered feature dictionary, search empirically: for each target `Local_feature`, which of a much wider set of raw wallet columns x aggregations correlates best?

In [ ]:
# Widen the candidate raw stats (18 wallet columns x 4 aggregations = 72
# candidates per transaction) and find, for each target Local_feature,
# whichever candidate correlates best -- an empirical search instead of
# trusting the recovered feature-dictionary text.

candidate_wallet_cols = [
    "num_txs_as_sender", "num_txs_as receiver", "total_txs",
    "lifetime_in_blocks", "num_timesteps_appeared_in",
    "btc_transacted_total", "btc_transacted_mean",
    "btc_sent_total", "btc_sent_mean",
    "btc_received_total", "btc_received_mean",
    "fees_total", "fees_mean",
    "blocks_btwn_txs_mean", "blocks_btwn_input_txs_mean", "blocks_btwn_output_txs_mean",
    "transacted_w_address_total", "transacted_w_address_mean",
]

wallet_stats_wide = wallets_features[
    ["address", "Time step"] + candidate_wallet_cols
].drop_duplicates(subset=["address", "Time step"])

input_wallet_wide = input_long[
    ["txId", "Time step", "address"]
].merge(wallet_stats_wide, on=["address", "Time step"], how="left")

agg_funcs = ["min", "max", "mean", "std"]
agg_dict = {
    f"{col}__{fn}": (col, fn)
    for col in candidate_wallet_cols
    for fn in agg_funcs
}

reconstructed_wide = input_wallet_wide.groupby("txId").agg(**agg_dict).reset_index()
print("Reconstructed wide feature table shape:", reconstructed_wide.shape)

target_local_features = [
    "Local_feature_53", "Local_feature_54", "Local_feature_55",
    "Local_feature_41", "Local_feature_42", "Local_feature_43",
    "Local_feature_47", "Local_feature_48", "Local_feature_49",
    "Local_feature_23", "Local_feature_24",
    "Local_feature_29", "Local_feature_30",
]

comparison_wide = reconstructed_wide.merge(
    tx_features[["txId"] + target_local_features], on="txId", how="inner"
)

candidate_cols = list(agg_dict.keys())
best_matches = []
for target in target_local_features:
    best_corr, best_candidate = 0.0, None
    for cand in candidate_cols:
        valid = comparison_wide[[cand, target]].dropna()
        if len(valid) < 100:
            continue
        corr = valid[cand].corr(valid[target], method="spearman")
        if pd.notna(corr) and abs(corr) > abs(best_corr):
            best_corr, best_candidate = corr, cand
    best_matches.append({
        "Local_feature": target,
        "Best_matching_raw_stat": best_candidate,
        "Spearman": round(best_corr, 4)
    })

best_match_df = pd.DataFrame(best_matches)
display(best_match_df)

Reconstructed wide feature table shape: (202804, 73)


   Local_feature         Best_matching_raw_stat  Spearman
0  Local_feature_53 num_timesteps_appeared_in__std    0.7140
1  Local_feature_54 num_timesteps_appeared_in__std    0.6253
2  Local_feature_55 num_timesteps_appeared_in__std    0.6980
3  Local_feature_41         num_txs_as_sender__max    0.8421
4  Local_feature_42 num_timesteps_appeared_in__std    0.7636
5  Local_feature_43         num_txs_as_sender__max    0.8414
6  Local_feature_47         num_txs_as_sender__max    0.8473
7  Local_feature_48 num_timesteps_appeared_in__std    0.7633
8  Local_feature_49         num_txs_as_sender__max    0.8466
9  Local_feature_23            btc_sent_total__max    0.8205
10 Local_feature_24      btc_transacted_total__std    0.7713
11 Local_feature_29            btc_sent_total__max    0.8206
12 Local_feature_30      btc_transacted_total__std    0.7714

**Key finding:** `Local_feature_41` (documented "min **incoming** transactions") and `Local_feature_47` (documented "min **outgoing** transactions") both best-match the *same* candidate, `num_txs_as_sender` (the outgoing-side count). That directly contradicts the documented incoming/outgoing labels.

Before concluding the dictionary is wrong, rule out the innocent explanation: maybe `num_txs_as_sender` and `num_txs_as receiver` are so collinear that no single-column search can tell them apart.

In [ ]:
# Check collinearity between num_txs_as_sender and num_txs_as receiver --
# if they're highly correlated, Step 3's identical best-match for both
# "incoming" and "outgoing" features could be a benign statistical artifact
# rather than a labeling error in the recovered feature dictionary.

pearson_sr = wallets_features["num_txs_as_sender"].corr(wallets_features["num_txs_as receiver"])
spearman_sr = wallets_features["num_txs_as_sender"].corr(
    wallets_features["num_txs_as receiver"], method="spearman"
)

print(f"num_txs_as_sender vs num_txs_as receiver -> Pearson={pearson_sr:.4f}  Spearman={spearman_sr:.4f}")

num_txs_as_sender vs num_txs_as receiver -> Pearson=0.4419  Spearman=-0.1116


**Not collinear** (Spearman -0.11, essentially unrelated by rank). So the label confusion in the previous cell is real, not a statistical artifact.

### Ceiling test: how much variance CAN be explained jointly?

Single-column matching is a weak test -- real construction might combine several raw columns nonlinearly. Fit a RandomForest regressor per target feature using all combined input+output candidate stats jointly, on a subsample with a held-out test split (same overfitting discipline as the main baseline models).

In [ ]:
# RF regression ceiling: how much of each target Local_feature's variance
# can be explained using ALL combined input+output wallet candidate stats
# jointly, rather than any single column? Subsampled to 25,000 transactions
# and capped trees/depth to keep this fast -- this is a diagnostic on
# feature reconstructability, not the main temporal experiment, so a random
# 80/20 split is fine here.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

output_addresses_long = tx_address_map[
    ["txId", "Time step", "output_addresses"]
].explode("output_addresses").rename(columns={"output_addresses": "address"})

sample_tx_ids = tx_features["txId"].sample(n=25000, random_state=42)

def reconstruct_wide_sampled(long_df, prefix, tx_ids):
    long_sub = long_df[long_df["txId"].isin(tx_ids)]
    merged = long_sub[["txId", "Time step", "address"]].merge(
        wallet_stats_wide, on=["address", "Time step"], how="left"
    )
    agg_dict_local = {f"{prefix}__{col}__{fn}": (col, fn) for col in candidate_wallet_cols for fn in agg_funcs}
    return merged.groupby("txId").agg(**agg_dict_local).reset_index()

wide_in_sample = reconstruct_wide_sampled(input_long, "in", sample_tx_ids)
wide_out_sample = reconstruct_wide_sampled(output_addresses_long, "out", sample_tx_ids)
wide_all_sample = wide_in_sample.merge(wide_out_sample, on="txId", how="outer")

ceiling_targets = ["Local_feature_53", "Local_feature_41", "Local_feature_47", "Local_feature_23"]
comparison_ceiling = wide_all_sample.merge(
    tx_features[["txId"] + ceiling_targets], on="txId", how="inner"
)
feature_cols = [c for c in wide_all_sample.columns if c != "txId"]

ceiling_results = []
for target in ceiling_targets:
    sub = comparison_ceiling[feature_cols + [target]].dropna(subset=[target])
    X = sub[feature_cols].fillna(-1)
    y = sub[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=80, max_depth=8, random_state=42, n_jobs=2)
    rf.fit(X_train, y_train)

    train_r2 = r2_score(y_train, rf.predict(X_train))
    test_r2 = r2_score(y_test, rf.predict(X_test))
    top_idx = np.argsort(rf.feature_importances_)[::-1][:3]
    top_feats = [(feature_cols[i], round(rf.feature_importances_[i], 3)) for i in top_idx]

    ceiling_results.append({
        "Local_feature": target, "Train_R2": round(train_r2, 4),
        "Test_R2": round(test_r2, 4), "Top_3_features": top_feats
    })
    print(f"{target}: train_R2={train_r2:.4f} test_R2={test_r2:.4f} top={top_feats}")

ceiling_df = pd.DataFrame(ceiling_results)

Local_feature_53: train_R2=0.8333 test_R2=0.8032 top=[('in__lifetime_in_blocks__max', 0.824), ('in__transacted_w_address_total__min', 0.008), ('in__fees_mean__std', 0.006)]
Local_feature_41: train_R2=0.9950 test_R2=0.9427 top=[('in__num_txs_as_sender__max', 0.327), ('in__num_timesteps_appeared_in__min', 0.249), ('in__total_txs__max', 0.046)]
Local_feature_47: train_R2=0.9971 test_R2=0.9745 top=[('in__num_txs_as_sender__max', 0.592), ('in__total_txs__max', 0.06), ('in__num_txs_as receiver__min', 0.041)]
Local_feature_23: train_R2=0.9978 test_R2=0.9833 top=[('in__btc_sent_total__max', 0.674), ('in__btc_received_total__max', 0.263), ('in__btc_transacted_total__max', 0.012)]


**Test R² of 0.80-0.98** using only combinations of wallet-level fields -- these features are substantially explainable from `wallets_features.csv`-level information, just not via the single documented column.

### The decisive check: do the driving columns even change over time?

This is the one that actually answers the transaction-time-availability question. Check whether the columns driving the R² above (`num_txs_as_sender`, `btc_sent_total`, `lifetime_in_blocks`, etc.) evolve across an address's time steps, or are frozen full-history values stamped at every time step.

In [ ]:
# For every wallet column that showed up as a top driver above, check
# whether it changes across an address's repeated time-step observations,
# or is a static full-history value copied to every time step the address
# appears in. This is the direct test of transaction-time availability.

cols_to_check = [
    "num_txs_as_sender", "num_txs_as receiver", "total_txs",
    "lifetime_in_blocks", "num_timesteps_appeared_in",
    "btc_sent_total", "btc_received_total", "btc_transacted_total",
    "first_block_appeared_in", "last_block_appeared_in",
]

obs_count = wallets_features.groupby("address").size()
multi_addresses = obs_count[obs_count > 1].index
wallets_multi = wallets_features[wallets_features["address"].isin(multi_addresses)]

print("Addresses with >1 observation:", len(multi_addresses))
print()

static_check_results = []
for col in cols_to_check:
    nun = wallets_multi.groupby("address")[col].nunique()
    pct_static = (nun == 1).mean() * 100
    static_check_results.append({"Column": col, "Pct_static_across_time_steps": round(pct_static, 2)})
    print(f"{col:30s} -> {pct_static:6.2f}% of repeated addresses have an IDENTICAL value at every time step")

static_check_df = pd.DataFrame(static_check_results)

# Show one concrete example: a high-activity address across many time steps
sample_addr = wallets_multi.groupby("address").size().idxmax()
example_cols = ["Time step"] + cols_to_check
example_rows = wallets_features[
    wallets_features["address"] == sample_addr
][example_cols].drop_duplicates().sort_values("Time step")

print(f"\nExample address: {sample_addr}  ({len(example_rows)} distinct time steps)")
display(example_rows)

Addresses with >1 observation: 273268

num_txs_as_sender              ->  100.00% of repeated addresses have an IDENTICAL value at every time step
num_txs_as receiver            ->  100.00% of repeated addresses have an IDENTICAL value at every time step
total_txs                      ->  100.00% of repeated addresses have an IDENTICAL value at every time step
lifetime_in_blocks             ->  100.00% of repeated addresses have an IDENTICAL value at every time step
num_timesteps_appeared_in      ->  100.00% of repeated addresses have an IDENTICAL value at every time step
btc_sent_total                 ->  100.00% of repeated addresses have an IDENTICAL value at every time step
btc_received_total             ->  100.00% of repeated addresses have an IDENTICAL value at every time step
btc_transacted_total           ->  100.00% of repeated addresses have an IDENTICAL value at every time step
first_block_appeared_in        ->  100.00% of repeated addresses have an IDENTICAL value at every

    Time step  num_txs_as_sender  num_txs_as receiver  total_txs  lifetime_in_blocks  num_timesteps_appeared_in  btc_sent_total  btc_received_total  btc_transacted_total  first_block_appeared_in  last_block_appeared_in
0          12              1453.0                  18.0     1471.0             54438.0                       27.0     1651.887703          2345.30849           3997.196193                 413376.0                467814.0
1          14              1453.0                  18.0     1471.0             54438.0                       27.0     1651.887703          2345.30849           3997.196193                 413376.0                467814.0
2          15              1453.0                  18.0     1471.0             54438.0                       27.0     1651.887703          2345.30849           3997.196193                 413376.0                467814.0
...        ...                 ...                   ...        ...                 ...                        ...    

### Conclusion so far (feature-construction / leakage audit)

**Chain of evidence:**
1. `wallets_features.csv`'s per-address fields (`num_txs_as_sender`, `btc_sent_total`, `lifetime_in_blocks`, etc.) are **100% static full-history snapshots**, not point-in-time state -- confirmed on 273,268 repeated addresses across 10 columns, with a concrete 27-time-step example showing zero variation.
2. The tested inherited `Local_feature_*` columns (53, 41, 47, 23) are **substantially explainable (test R² 0.80-0.98)** by combinations of those same static fields -- not through the single documented raw-column mapping (which mostly failed in the earlier cells), but through weighted combinations.
3. **Therefore:** these features very likely encode full-history address information rather than information available strictly at the transaction's own point in time.

**Scope of this claim:** this is strong, self-generated, reproducible empirical evidence -- not literal proof of the original Elliptic++ construction code (which we don't have access to), and only 4 of the ~166 inherited features (53, 41, 47, 23) received the full R² + static-field treatment. Extending this to the rest of the address-history feature family (from the earlier RF importance audit: 52, 61, 65, 90, 25, etc.) is the natural next step before finalizing the leakage-audit chapter.

**Also documented for reproducibility:** an early version of the R² ceiling test (144 features, full 200K-row dataset, unthrottled `n_jobs=-1`) ran for 77 minutes without completing on this machine and was killed; the version above (25,000-row subsample, capped trees/depth, `n_jobs=2`) reproduces the same conclusion in under 3 minutes.